In [3]:
from pathlib import Path
import os
import sys
import json
import time

import numpy as np
import pandas as pd
import lightgbm as lgb

REPO = Path.home() / "Amazon-ML-Challenge"
CODE_ROOT = REPO / "code" / "business_entity_resolution"
SRC = CODE_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

os.chdir(REPO)

print("Repo:", REPO)
print("Python:", sys.version.split()[0])
print("LightGBM:", lgb.__version__)
print("Working directory:", Path.cwd())

Repo: /home/jupyter-1nt23cb058/Amazon-ML-Challenge
Python: 3.9.7
LightGBM: 4.6.0
Working directory: /home/jupyter-1nt23cb058/Amazon-ML-Challenge


In [4]:
from collections import defaultdict

search_roots = [
    REPO / "cache",
    REPO / "models",
    REPO / "experiments",
]

patterns = ["*.parquet", "*.csv", "*.tsv", "*.npy", "*.npz", "*.json", "*.txt"]

found = []

for root in search_roots:
    if not root.exists():
        continue

    for pattern in patterns:
        for p in root.rglob(pattern):
            found.append(
                (
                    str(p.relative_to(REPO)),
                    p.stat().st_size / (1024**2),
                    p.stat().st_mtime,
                )
            )

found = sorted(found, key=lambda x: x[2], reverse=True)

print(f"Found {len(found)} artifacts\n")

for path, size_mb, mtime in found[:100]:
    print(f"{size_mb:9.2f} MB   {path}")

Found 64 artifacts

     0.07 MB   experiments/e23_retrieval_38a0b640f78b_metrics.json
     0.00 MB   cache/candidates/e23_retrieval_38a0b640f78b_selected.parquet.meta.json
    10.22 MB   cache/candidates/e23_retrieval_38a0b640f78b_selected.parquet
     0.00 MB   cache/candidates/e23_retrieval_38a0b640f78b_address_tfidf.parquet.meta.json
     3.19 MB   cache/candidates/e23_retrieval_38a0b640f78b_address_tfidf.parquet
     0.07 MB   experiments/e23_retrieval_bb2d7121ad1c_metrics.json
    10.22 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_selected.parquet
     0.00 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_selected.parquet.meta.json
     3.19 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_address_tfidf.parquet
     0.00 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_address_tfidf.parquet.meta.json
     0.10 MB   experiments/e22_retrieval_e7c71a736237_metrics.json
     0.00 MB   cache/candidates/e22_retrieval_e7c71a736237_selected.parquet.meta.json
     9.80 MB   cache/cand

In [5]:
keywords = [
    "e2",
    "e21",
    "e22",
    "e23",
    "candidate",
    "feature",
    "prediction",
    "validation",
    "train",
]

interesting = []

for path, size_mb, mtime in found:
    lower = path.lower()
    if any(k in lower for k in keywords):
        interesting.append((path, size_mb))

for path, size_mb in interesting[:100]:
    print(f"{size_mb:9.2f} MB   {path}")

     0.07 MB   experiments/e23_retrieval_38a0b640f78b_metrics.json
     0.00 MB   cache/candidates/e23_retrieval_38a0b640f78b_selected.parquet.meta.json
    10.22 MB   cache/candidates/e23_retrieval_38a0b640f78b_selected.parquet
     0.00 MB   cache/candidates/e23_retrieval_38a0b640f78b_address_tfidf.parquet.meta.json
     3.19 MB   cache/candidates/e23_retrieval_38a0b640f78b_address_tfidf.parquet
     0.07 MB   experiments/e23_retrieval_bb2d7121ad1c_metrics.json
    10.22 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_selected.parquet
     0.00 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_selected.parquet.meta.json
     3.19 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_address_tfidf.parquet
     0.00 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_address_tfidf.parquet.meta.json
     0.10 MB   experiments/e22_retrieval_e7c71a736237_metrics.json
     0.00 MB   cache/candidates/e22_retrieval_e7c71a736237_selected.parquet.meta.json
     9.80 MB   cache/candidates/e22_retrieval

In [6]:
from pathlib import Path
import json
import pprint

E23_RUNS = [
    "e23_retrieval_38a0b640f78b",
    "e23_retrieval_bb2d7121ad1c",
]

for run in E23_RUNS:
    path = REPO / "experiments" / f"{run}_metrics.json"
    print("\n" + "=" * 100)
    print(run)
    print("=" * 100)

    with open(path) as f:
        metrics = json.load(f)

    pprint.pp(metrics, depth=4, width=120)


e23_retrieval_38a0b640f78b
{'artifact_id': '38a0b640f78b',
 'artifacts': {'optimized_address_candidates': '/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/candidates/e23_retrieval_38a0b640f78b_address_tfidf.parquet',
               'report': '/home/jupyter-1nt23cb058/Amazon-ML-Challenge/experiments/e23_retrieval_38a0b640f78b_metrics.json',
               'selected_candidates': '/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/candidates/e23_retrieval_38a0b640f78b_selected.parquet'},
 'cache_disk_usage_bytes': {'new_candidate_caches_total': 14060027,
                            'optimized_address_candidates': 3347642,
                            'selected_candidates': 10712385,
                            'sparse_index_cached': False},
 'cache_reuse': {'e21_candidates': '/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/candidates/e21_lgbm_c449c0c6b281_candidates.parquet',
                 'feed_rows': 1034502,
                 'legacy_candidates': '/home/jupyter-1nt23cb058/Amazo

                                                  {'address_tfidf_rank': 9,
                                                   'address_tfidf_similarity': 0.7342166900634766,
                                                   'candidate_address': 'Sacramento, 07916 Valley Green Dr, California',
                                                   'candidate_entity_id': 'S3-638659610',
                                                   'candidate_name': 'Bexi Lng',
                                                   'country': 'us',
                                                   'failure_category': 'other',
                                                   'routes': {},
                                                   'source1_address': '7916 Valley Green Drive, Sacramento, CA',
                                                   'source1_entity_id': 'S1-369212908',
                                                   'source1_name': 'Beix Lng'},
                                       

In [8]:
for run in E23_RUNS:
    path = REPO / "cache" / "candidates" / f"{run}_selected.parquet"
    df = pd.read_parquet(path)

    print("\n", run)
    print("shape:", df.shape)
    print("columns:")
    print(df.columns.tolist())
    display(df.head(3))


 e23_retrieval_38a0b640f78b
shape: (488090, 45)
columns:
['source1_entity_id', 'candidate_entity_id', 'candidate_source', 'same_country_block', 'cheap_score', 'cheap_rank', 'cheap_score_margin_to_next', 'rare_name_token_hits', 'rare_address_token_hits', 'block_exact_full_name', 'block_exact_core_name', 'block_exact_sorted_name', 'block_rare_name_token', 'block_exact_address', 'block_rare_address_token', 'block_cross_country_exact_name', 'has_legacy', 'name_tfidf_similarity', 'name_tfidf_rank', 'has_tfidf', 'legacy_rank', 'legacy_cheap_score', 'block_name_tfidf', 'is_tfidf_only', 'is_legacy_only', 'supported_by_both', 'legacy_blocking_channel_count', 'candidate_count_raw', 'address_tfidf_similarity', 'address_tfidf_rank', 'address_route_postcode', 'address_route_postcode_prefix', 'address_route_numeric', 'address_route_rare_token', 'address_route_country_fallback', 'address_route_count', 'address_preblock_pool_size', 'has_address_tfidf', 'block_address_tfidf', 'is_address_tfidf_only', 

,source1_entity_id,candidate_entity_id,candidate_source,same_country_block,cheap_score,cheap_rank,cheap_score_margin_to_next,rare_name_token_hits,rare_address_token_hits,block_exact_full_name,...,address_route_count,address_preblock_pool_size,has_address_tfidf,block_address_tfidf,is_address_tfidf_only,retrieval_channel_count,fusion_score,fusion_rank,candidate_count_for_s1,fusion_strategy
0,S1-100077788,S3-490363076,S3,1,15.0,1,5.0,1.0,0.0,1.0,...,1,177,1,1,0,3,0.190476,1,50,protected35_lw2_nw1_aw1_c20
1,S1-100077788,S3-771950070,S3,1,10.0,2,8.5,1.0,0.0,0.0,...,1,177,1,1,0,3,0.181818,2,50,protected35_lw2_nw1_aw1_c20
2,S1-100077788,S2-454666788,S2,1,1.5,3,0.0,1.0,0.0,0.0,...,0,0,0,0,0,2,0.120401,3,50,protected35_lw2_nw1_aw1_c20



 e23_retrieval_bb2d7121ad1c
shape: (488091, 45)
columns:
['source1_entity_id', 'candidate_entity_id', 'candidate_source', 'same_country_block', 'cheap_score', 'cheap_rank', 'cheap_score_margin_to_next', 'rare_name_token_hits', 'rare_address_token_hits', 'block_exact_full_name', 'block_exact_core_name', 'block_exact_sorted_name', 'block_rare_name_token', 'block_exact_address', 'block_rare_address_token', 'block_cross_country_exact_name', 'has_legacy', 'name_tfidf_similarity', 'name_tfidf_rank', 'has_tfidf', 'legacy_rank', 'legacy_cheap_score', 'block_name_tfidf', 'is_tfidf_only', 'is_legacy_only', 'supported_by_both', 'legacy_blocking_channel_count', 'candidate_count_raw', 'address_tfidf_similarity', 'address_tfidf_rank', 'address_route_postcode', 'address_route_postcode_prefix', 'address_route_numeric', 'address_route_rare_token', 'address_route_country_fallback', 'address_route_count', 'address_preblock_pool_size', 'has_address_tfidf', 'block_address_tfidf', 'is_address_tfidf_only', 

,source1_entity_id,candidate_entity_id,candidate_source,same_country_block,cheap_score,cheap_rank,cheap_score_margin_to_next,rare_name_token_hits,rare_address_token_hits,block_exact_full_name,...,address_route_count,address_preblock_pool_size,has_address_tfidf,block_address_tfidf,is_address_tfidf_only,retrieval_channel_count,fusion_score,fusion_rank,candidate_count_for_s1,fusion_strategy
0,S1-100077788,S3-490363076,S3,1,15.0,1,5.0,1.0,0.0,1.0,...,1,177,1,1,0,3,0.190476,1,50,protected35_lw2_nw1_aw1_c20
1,S1-100077788,S3-771950070,S3,1,10.0,2,8.5,1.0,0.0,0.0,...,1,177,1,1,0,3,0.181818,2,50,protected35_lw2_nw1_aw1_c20
2,S1-100077788,S2-454666788,S2,1,1.5,3,0.0,1.0,0.0,0.0,...,0,0,0,0,0,2,0.120401,3,50,protected35_lw2_nw1_aw1_c20


In [9]:
OLD_FEATURE_PATH = (
    REPO
    / "cache"
    / "features"
    / "e21_lgbm_c449c0c6b281_features.parquet"
)

old_features = pd.read_parquet(OLD_FEATURE_PATH)

print("Old E2.1 feature table:", old_features.shape)
print("\nColumns:")
for i, col in enumerate(old_features.columns):
    print(f"{i:02d}: {col}")

display(old_features.head(3))

Old E2.1 feature table: (403984, 63)

Columns:
00: source1_entity_id
01: candidate_entity_id
02: name_full_exact
03: name_core_exact
04: name_sorted_exact
05: name_token_jaccard
06: name_shared_token_count
07: name_token_count_difference
08: name_length_ratio
09: name_character_similarity
10: name_prefix_ratio
11: name_suffix_ratio
12: address_exact
13: address_token_jaccard
14: address_shared_token_count
15: numeric_token_equal
16: numeric_token_overlap
17: numeric_token_conflict
18: address_length_ratio
19: address_missing_left
20: address_missing_right
21: same_country
22: country_conflict
23: country_missing_left
24: country_missing_right
25: both_country_present
26: source_is_s2
27: source_is_s3
28: block_exact_full_name
29: block_exact_core_name
30: block_exact_sorted_name
31: block_rare_name_token
32: block_exact_address
33: block_rare_address_token
34: block_cross_country_exact_name
35: block_name_tfidf
36: blocking_channel_count
37: rare_name_token_hits
38: rare_address_token_

,source1_entity_id,candidate_entity_id,name_full_exact,name_core_exact,name_sorted_exact,name_token_jaccard,name_shared_token_count,name_token_count_difference,name_length_ratio,name_character_similarity,...,label,relative_name_similarity_rank,relative_address_similarity_rank,relative_tfidf_similarity_rank,relative_fusion_score_rank,top_name_similarity_margin,top_address_similarity_margin,top_tfidf_similarity_margin,fusion_score_gap_to_best,close_name_competitor_count
0,S1-100077788,S3-490363076,1,1,1,1.00,4,0,1.000000,1.000000,...,1,1,2,1,1,0.020408,0.036364,0.0,0.000000,2
1,S1-100077788,S3-771950070,0,1,1,0.75,3,1,0.833333,0.909091,...,1,3,1,2,2,0.020408,0.036364,0.0,0.006494,2
2,S1-100077788,S2-454666788,0,0,0,0.60,3,0,0.960000,0.979592,...,0,2,14,3,3,0.020408,0.036364,0.0,0.022456,2


In [10]:
with open(
    REPO / "models" / "lgbm" / "e21_lgbm_c449c0c6b281_features.json"
) as f:
    old_feature_spec = json.load(f)

old_feature_spec

['name_full_exact',
 'name_core_exact',
 'name_sorted_exact',
 'name_token_jaccard',
 'name_shared_token_count',
 'name_token_count_difference',
 'name_length_ratio',
 'name_character_similarity',
 'name_prefix_ratio',
 'name_suffix_ratio',
 'address_exact',
 'address_token_jaccard',
 'address_shared_token_count',
 'numeric_token_equal',
 'numeric_token_overlap',
 'numeric_token_conflict',
 'address_length_ratio',
 'address_missing_left',
 'address_missing_right',
 'same_country',
 'country_conflict',
 'country_missing_left',
 'country_missing_right',
 'both_country_present',
 'source_is_s2',
 'source_is_s3',
 'block_exact_full_name',
 'block_exact_core_name',
 'block_exact_sorted_name',
 'block_rare_name_token',
 'block_exact_address',
 'block_rare_address_token',
 'block_cross_country_exact_name',
 'block_name_tfidf',
 'blocking_channel_count',
 'rare_name_token_hits',
 'rare_address_token_hits',
 'name_tfidf_similarity',
 'name_tfidf_rank',
 'legacy_blocking_channel_count',
 'legacy

In [11]:
import inspect

import business_entity_resolution.features as feature_module

print(inspect.getsource(feature_module)[:30000])

"""Compact, LightGBM-ready numeric features for blocked candidate pairs."""

from __future__ import annotations

from difflib import SequenceMatcher
from typing import Dict, Iterator, List, Set

import numpy as np
import pandas as pd

from .blocking import PROVENANCE_COLUMNS


IDENTIFIER_COLUMNS = ["source1_entity_id", "candidate_entity_id"]

FEATURE_COLUMNS = [
    "name_full_exact",
    "name_core_exact",
    "name_sorted_exact",
    "name_token_jaccard",
    "name_shared_token_count",
    "name_token_count_difference",
    "name_length_ratio",
    "name_character_similarity",
    "name_prefix_ratio",
    "name_suffix_ratio",
    "address_exact",
    "address_token_jaccard",
    "address_shared_token_count",
    "numeric_token_equal",
    "numeric_token_overlap",
    "numeric_token_conflict",
    "address_length_ratio",
    "address_missing_left",
    "address_missing_right",
    "same_country",
    "country_conflict",
    "country_missing_left",
    "country_missing_right",
    "bot

In [12]:
# Cell 9: Load the selected E2.3 candidate set and the normalized Source1/feed records
# that we will use to build pairwise model features.

RUN = "e23_retrieval_38a0b640f78b"

CANDIDATE_PATH = (
    REPO / "cache" / "candidates" / f"{RUN}_selected.parquet"
)

SOURCE1_PATH = (
    REPO
    / "cache"
    / "normalized"
    / "e21_lgbm_c449c0c6b281_source1.parquet"
)

FEED_PATH = (
    REPO
    / "cache"
    / "normalized"
    / "e21_lgbm_c449c0c6b281_feed.parquet"
)

candidates = pd.read_parquet(CANDIDATE_PATH)
source1 = pd.read_parquet(SOURCE1_PATH)
feed = pd.read_parquet(FEED_PATH)

print("Candidates:", candidates.shape)
print("Source1:", source1.shape)
print("Feed:", feed.shape)

assert len(candidates) == 488090
assert candidates["source1_entity_id"].nunique() == 10_000

print("✅ E2.3 data loaded")

Candidates: (488090, 45)
Source1: (10000, 18)
Feed: (1034502, 23)
✅ E2.3 data loaded


In [13]:
# Cell 10: Build the base pairwise feature table for all E2.3 candidate pairs
# using the existing feature-engineering pipeline. This is CPU feature generation,
# not model training yet.

from business_entity_resolution.features import build_feature_table

t0 = time.time()

features = build_feature_table(
    candidates,
    source1,
    feed,
    chunk_size=50_000,
)

elapsed = time.time() - t0

print(f"Base features generated in {elapsed:.1f}s")
print("Feature table shape:", features.shape)
print("Feature columns:", len(features.columns))

display(features.head(3))

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [14]:
# Cell 10A: Identify which candidate columns contain NaN/inf values that break
# the old feature builder's integer downcasting. This cell only inspects data.

problem_cols = []

for col in candidates.columns:
    if pd.api.types.is_numeric_dtype(candidates[col]):
        n_nan = candidates[col].isna().sum()
        n_inf = np.isinf(
            pd.to_numeric(candidates[col], errors="coerce").fillna(0)
        ).sum()

        if n_nan > 0 or n_inf > 0:
            problem_cols.append((col, int(n_nan), int(n_inf)))

print("Columns with NaN/inf:")
for col, n_nan, n_inf in problem_cols:
    print(f"{col:40s} NaN={n_nan:8d}  inf={n_inf:8d}")

print("\nTotal problematic columns:", len(problem_cols))

Columns with NaN/inf:
cheap_score                              NaN=  111314  inf=       0
cheap_score_margin_to_next               NaN=  111314  inf=       0
rare_name_token_hits                     NaN=  111314  inf=       0
rare_address_token_hits                  NaN=  111314  inf=       0
block_exact_full_name                    NaN=  111314  inf=       0
block_exact_core_name                    NaN=  111314  inf=       0
block_exact_sorted_name                  NaN=  111314  inf=       0
block_rare_name_token                    NaN=  111314  inf=       0
block_exact_address                      NaN=  111314  inf=       0
block_rare_address_token                 NaN=  111314  inf=       0
block_cross_country_exact_name           NaN=  111314  inf=       0
legacy_cheap_score                       NaN=  111314  inf=       0
block_name_tfidf                         NaN=  111314  inf=       0
is_tfidf_only                            NaN=  111314  inf=       0
is_legacy_only            

In [15]:
# Cell 10B: Fill missing legacy/name-retrieval metadata with neutral defaults
# for E2.3 address-only candidates so the old feature builder can process them.

candidates_clean = candidates.copy()

ZERO_FILL_COLS = [
    "cheap_score",
    "cheap_score_margin_to_next",
    "rare_name_token_hits",
    "rare_address_token_hits",
    "block_exact_full_name",
    "block_exact_core_name",
    "block_exact_sorted_name",
    "block_rare_name_token",
    "block_exact_address",
    "block_rare_address_token",
    "block_cross_country_exact_name",
    "legacy_cheap_score",
    "block_name_tfidf",
    "is_tfidf_only",
    "is_legacy_only",
    "supported_by_both",
    "legacy_blocking_channel_count",
    "candidate_count_raw",
]

for col in ZERO_FILL_COLS:
    candidates_clean[col] = candidates_clean[col].fillna(0)

print("Remaining NaNs in cleaned candidate table:")
remaining = candidates_clean[ZERO_FILL_COLS].isna().sum()
print(remaining[remaining > 0])

print("\nRows cleaned:", (candidates[ZERO_FILL_COLS].isna().any(axis=1)).sum())
print("✅ Legacy/name metadata cleaned")

Remaining NaNs in cleaned candidate table:
Series([], dtype: int64)

Rows cleaned: 111314
✅ Legacy/name metadata cleaned


In [16]:
# Cell 10C: Rebuild the base pairwise feature table using the cleaned E2.3
# candidate metadata. This is still feature engineering, not model training.

import time

t0 = time.time()

features = build_feature_table(
    candidates_clean,
    source1,
    feed,
    chunk_size=50_000,
)

elapsed = time.time() - t0

print(f"Base features generated in {elapsed:.1f}s")
print("Feature table shape:", features.shape)
print("Feature columns:", len(features.columns))

display(features.head(3))

Base features generated in 229.0s
Feature table shape: (488090, 53)
Feature columns: 53


,source1_entity_id,candidate_entity_id,name_full_exact,name_core_exact,name_sorted_exact,name_token_jaccard,name_shared_token_count,name_token_count_difference,name_length_ratio,name_character_similarity,...,legacy_cheap_score,is_tfidf_only,is_legacy_only,supported_by_both,fusion_score,fusion_rank,candidate_count_for_s1,cheap_score,cheap_rank,cheap_score_margin_to_next
0,S1-100077788,S3-490363076,1,1,1,1.00,4,0,1.000000,1.000000,...,15.0,0,0,1,0.190476,1,50,15.0,1,5.0
1,S1-100077788,S3-771950070,0,1,1,0.75,3,1,0.833333,0.909091,...,10.0,0,0,1,0.181818,2,50,10.0,2,8.5
2,S1-100077788,S2-454666788,0,0,0,0.60,3,0,0.960000,0.979592,...,1.5,0,0,1,0.120401,3,50,1.5,3,0.0


In [17]:
# Cell 11: Append E2.3 address-retrieval metadata to the base pairwise feature table.
# These are retrieval/provenance signals that were not part of the old E2.1 feature builder.

E23_EXTRA_COLUMNS = [
    "source1_entity_id",
    "candidate_entity_id",
    "address_tfidf_similarity",
    "address_tfidf_rank",
    "address_route_postcode",
    "address_route_postcode_prefix",
    "address_route_numeric",
    "address_route_rare_token",
    "address_route_country_fallback",
    "address_route_count",
    "address_preblock_pool_size",
    "has_address_tfidf",
    "block_address_tfidf",
    "is_address_tfidf_only",
    "retrieval_channel_count",
]

extra = candidates_clean[E23_EXTRA_COLUMNS].copy()

features = features.merge(
    extra,
    on=["source1_entity_id", "candidate_entity_id"],
    how="left",
    validate="one_to_one",
)

# Compact dtypes to keep memory under control.
binary_cols = [
    "address_route_postcode",
    "address_route_postcode_prefix",
    "address_route_numeric",
    "address_route_rare_token",
    "address_route_country_fallback",
    "has_address_tfidf",
    "block_address_tfidf",
    "is_address_tfidf_only",
]

for col in binary_cols:
    features[col] = features[col].fillna(0).astype("uint8")

features["address_tfidf_similarity"] = (
    features["address_tfidf_similarity"].fillna(0).astype("float32")
)

count_cols = [
    "address_tfidf_rank",
    "address_route_count",
    "address_preblock_pool_size",
    "retrieval_channel_count",
]

for col in count_cols:
    features[col] = features[col].fillna(0).astype("uint32")

print("Feature table shape:", features.shape)
print("Columns:", len(features.columns))
print("Remaining NaNs in E2.3 extras:",
      int(features[E23_EXTRA_COLUMNS[2:]].isna().sum().sum()))

display(
    features[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "address_tfidf_similarity",
            "address_tfidf_rank",
            "has_address_tfidf",
            "is_address_tfidf_only",
            "retrieval_channel_count",
        ]
    ].head(5)
)

Feature table shape: (488090, 66)
Columns: 66
Remaining NaNs in E2.3 extras: 0


,source1_entity_id,candidate_entity_id,address_tfidf_similarity,address_tfidf_rank,has_address_tfidf,is_address_tfidf_only,retrieval_channel_count
0,S1-100077788,S3-490363076,0.725058,1,1,0,3
1,S1-100077788,S3-771950070,0.713299,2,1,0,3
2,S1-100077788,S2-454666788,0.000000,0,0,0,2
3,S1-100077788,S3-291121444,0.000000,0,0,0,2
4,S1-100077788,S2-204267984,0.000000,0,0,0,1


In [18]:
# Cell 12: Add the existing candidate-relative features such as within-entity
# similarity ranks and score margins used by the previous LightGBM model.

from business_entity_resolution.features import add_candidate_relative_features

t0 = time.time()

features = add_candidate_relative_features(features)

elapsed = time.time() - t0

print(f"Relative features added in {elapsed:.1f}s")
print("Feature table shape:", features.shape)
print("Columns:", len(features.columns))

display(
    features[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "relative_name_similarity_rank",
            "relative_address_similarity_rank",
            "relative_tfidf_similarity_rank",
            "relative_fusion_score_rank",
            "top_name_similarity_margin",
            "top_address_similarity_margin",
            "top_tfidf_similarity_margin",
            "fusion_score_gap_to_best",
            "close_name_competitor_count",
        ]
    ].head(5)
)

Relative features added in 5.0s
Feature table shape: (488090, 75)
Columns: 75


,source1_entity_id,candidate_entity_id,relative_name_similarity_rank,relative_address_similarity_rank,relative_tfidf_similarity_rank,relative_fusion_score_rank,top_name_similarity_margin,top_address_similarity_margin,top_tfidf_similarity_margin,fusion_score_gap_to_best,close_name_competitor_count
0,S1-100077788,S3-490363076,1,4,1,1,0.020408,0.036364,0.0,0.000000,2
1,S1-100077788,S3-771950070,3,1,2,2,0.020408,0.036364,0.0,0.008658,2
2,S1-100077788,S2-454666788,2,27,3,3,0.020408,0.036364,0.0,0.070075,2
3,S1-100077788,S3-291121444,23,38,5,4,0.020408,0.036364,0.0,0.096422,2
4,S1-100077788,S2-204267984,31,21,22,5,0.020408,0.036364,0.0,0.103520,2


In [19]:
# Cell 13: Add within-entity relative features for the new address-TFIDF signal,
# so LightGBM can use both absolute similarity and how strong it is versus competitors.

from business_entity_resolution.features import _deterministic_descending_rank

features["relative_address_tfidf_rank"] = (
    _deterministic_descending_rank(
        features,
        "address_tfidf_similarity",
    ).astype("uint32")
)

best_address_tfidf = (
    features
    .groupby("source1_entity_id")["address_tfidf_similarity"]
    .transform("max")
)

features["address_tfidf_gap_to_best"] = (
    best_address_tfidf - features["address_tfidf_similarity"]
).astype("float32")

print("Feature table shape:", features.shape)
print("Columns:", len(features.columns))

display(
    features[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "address_tfidf_similarity",
            "relative_address_tfidf_rank",
            "address_tfidf_gap_to_best",
        ]
    ].head(10)
)

Feature table shape: (488090, 77)
Columns: 77


,source1_entity_id,candidate_entity_id,address_tfidf_similarity,relative_address_tfidf_rank,address_tfidf_gap_to_best
0,S1-100077788,S3-490363076,0.725058,1,0.000000
1,S1-100077788,S3-771950070,0.713299,2,0.011759
2,S1-100077788,S2-454666788,0.000000,24,0.725058
3,S1-100077788,S3-291121444,0.000000,37,0.725058
4,S1-100077788,S2-204267984,0.000000,18,0.725058
5,S1-100077788,S2-233535286,0.000000,19,0.725058
6,S1-100077788,S2-326312152,0.000000,20,0.725058
7,S1-100077788,S2-634900154,0.000000,25,0.725058
8,S1-100077788,S3-408233518,0.000000,39,0.725058
9,S1-100077788,S3-433788926,0.000000,40,0.725058


In [20]:
# Cell 14: Locate the training ground-truth file and inspect its columns/schema
# before converting true entity matches into binary candidate-pair labels.

from pathlib import Path

gt_candidates = list(
    (REPO / "student_resource").rglob("*ground*truth*.tsv")
)

print("Ground-truth files found:")
for p in gt_candidates:
    print(" -", p)

assert gt_candidates, "No ground-truth TSV found under student_resource"

GT_PATH = gt_candidates[0]

gt = pd.read_csv(GT_PATH, sep="\t")

print("\nUsing:", GT_PATH)
print("Shape:", gt.shape)
print("Columns:", gt.columns.tolist())

display(gt.head(5))

Ground-truth files found:
 - /home/jupyter-1nt23cb058/Amazon-ML-Challenge/student_resource/dataset/train/train_ground_truth.tsv

Using: /home/jupyter-1nt23cb058/Amazon-ML-Challenge/student_resource/dataset/train/train_ground_truth.tsv
Shape: (2206821, 2)
Columns: ['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [21]:
# Cell 15: Build binary labels for the 488,090 candidate pairs by filtering the
# ground truth to our 10,000 Source1 entities, exploding true matches, and joining.

experiment_s1_ids = set(features["source1_entity_id"].unique())

gt_subset = gt[
    gt["source1_entity_id"].isin(experiment_s1_ids)
].copy()

# Turn comma-separated matched IDs into one true pair per row.
true_pairs = (
    gt_subset
    .assign(
        candidate_entity_id=
        gt_subset["matched_entity_ids"]
        .fillna("")
        .astype(str)
        .str.split(",")
    )
    .explode("candidate_entity_id")
)

# Remove empty-match rows.
true_pairs["candidate_entity_id"] = (
    true_pairs["candidate_entity_id"].astype(str).str.strip()
)

true_pairs = true_pairs[
    true_pairs["candidate_entity_id"] != ""
][["source1_entity_id", "candidate_entity_id"]].drop_duplicates()

true_pairs["label"] = 1

# Join labels onto candidate feature table; all non-true candidates become negatives.
features = features.merge(
    true_pairs,
    on=["source1_entity_id", "candidate_entity_id"],
    how="left",
    validate="one_to_one",
)

features["label"] = features["label"].fillna(0).astype("uint8")

print("Experiment Source1 entities:", len(experiment_s1_ids))
print("True links in ground truth subset:", len(true_pairs))
print("Positive candidate pairs recovered:", int(features["label"].sum()))
print("Negative candidate pairs:", int((features["label"] == 0).sum()))
print("Positive rate:", f"{features['label'].mean():.4%}")
print("Feature table shape:", features.shape)

assert features["source1_entity_id"].nunique() == 10_000
assert len(features) == 488_090

print("✅ Labels attached")

Experiment Source1 entities: 10000
True links in ground truth subset: 34502
Positive candidate pairs recovered: 33667
Negative candidate pairs: 454423
Positive rate: 6.8977%
Feature table shape: (488090, 78)
✅ Labels attached


In [22]:
# Cell 16: Recreate the exact E2.1/E2.3 train/validation split by loading the
# previously saved validation predictions and extracting their Source1 entity IDs.

OLD_VAL_PRED_PATH = (
    REPO
    / "cache"
    / "predictions"
    / "e21_lgbm_c449c0c6b281_validation.parquet"
)

old_val_predictions = pd.read_parquet(OLD_VAL_PRED_PATH)

val_s1_ids = set(
    old_val_predictions["source1_entity_id"].unique()
)

all_s1_ids = set(
    features["source1_entity_id"].unique()
)

train_s1_ids = all_s1_ids - val_s1_ids

print("Train Source1 entities:", len(train_s1_ids))
print("Validation Source1 entities:", len(val_s1_ids))
print("Total:", len(train_s1_ids) + len(val_s1_ids))

assert len(train_s1_ids) == 7568
assert len(val_s1_ids) == 2432
assert train_s1_ids.isdisjoint(val_s1_ids)

print("✅ Exact previous entity split recovered")

Train Source1 entities: 7568
Validation Source1 entities: 2432
Total: 10000
✅ Exact previous entity split recovered


In [23]:
# Cell 17: Build the final LightGBM feature list and create train/validation
# matrices using the exact entity-level split. This prepares training only.

OLD_MODEL_FEATURES = [
    "name_full_exact",
    "name_core_exact",
    "name_sorted_exact",
    "name_token_jaccard",
    "name_shared_token_count",
    "name_token_count_difference",
    "name_length_ratio",
    "name_character_similarity",
    "name_prefix_ratio",
    "name_suffix_ratio",
    "address_exact",
    "address_token_jaccard",
    "address_shared_token_count",
    "numeric_token_equal",
    "numeric_token_overlap",
    "numeric_token_conflict",
    "address_length_ratio",
    "address_missing_left",
    "address_missing_right",
    "same_country",
    "country_conflict",
    "country_missing_left",
    "country_missing_right",
    "both_country_present",
    "source_is_s2",
    "source_is_s3",
    "block_exact_full_name",
    "block_exact_core_name",
    "block_exact_sorted_name",
    "block_rare_name_token",
    "block_exact_address",
    "block_rare_address_token",
    "block_cross_country_exact_name",
    "block_name_tfidf",
    "blocking_channel_count",
    "rare_name_token_hits",
    "rare_address_token_hits",
    "name_tfidf_similarity",
    "name_tfidf_rank",
    "legacy_blocking_channel_count",
    "legacy_rank",
    "legacy_cheap_score",
    "is_tfidf_only",
    "is_legacy_only",
    "supported_by_both",
    "fusion_score",
    "fusion_rank",
    "candidate_count_for_s1",
    "cheap_score",
    "cheap_rank",
    "cheap_score_margin_to_next",
    "relative_name_similarity_rank",
    "relative_address_similarity_rank",
    "relative_tfidf_similarity_rank",
    "relative_fusion_score_rank",
    "top_name_similarity_margin",
    "top_address_similarity_margin",
    "top_tfidf_similarity_margin",
    "fusion_score_gap_to_best",
    "close_name_competitor_count",
]

E23_MODEL_FEATURES = [
    "address_tfidf_similarity",
    "address_tfidf_rank",
    "address_route_postcode",
    "address_route_postcode_prefix",
    "address_route_numeric",
    "address_route_rare_token",
    "address_route_country_fallback",
    "address_route_count",
    "address_preblock_pool_size",
    "has_address_tfidf",
    "block_address_tfidf",
    "is_address_tfidf_only",
    "retrieval_channel_count",
    "relative_address_tfidf_rank",
    "address_tfidf_gap_to_best",
]

model_features = OLD_MODEL_FEATURES + E23_MODEL_FEATURES

missing = [c for c in model_features if c not in features.columns]
assert not missing, f"Missing model features: {missing}"

train_mask = features["source1_entity_id"].isin(train_s1_ids)
val_mask = features["source1_entity_id"].isin(val_s1_ids)

X_train = features.loc[train_mask, model_features]
y_train = features.loc[train_mask, "label"]

X_val = features.loc[val_mask, model_features]
y_val = features.loc[val_mask, "label"]

print("Model features:", len(model_features))
print("Train pairs:", len(X_train))
print("Validation pairs:", len(X_val))
print("Train positives:", int(y_train.sum()))
print("Validation positives:", int(y_val.sum()))
print("Train positive rate:", f"{y_train.mean():.4%}")
print("Validation positive rate:", f"{y_val.mean():.4%}")

assert X_train.shape[1] == 75
assert X_val.shape[1] == 75

print("✅ LightGBM matrices ready")

Model features: 75
Train pairs: 369671
Validation pairs: 118419
Train positives: 25404
Validation positives: 8263
Train positive rate: 6.8721%
Validation positive rate: 6.9778%
✅ LightGBM matrices ready


In [24]:
# Cell 18: Save a durable checkpoint of the completed E2.3 feature table,
# labels, exact train/validation split, and LightGBM feature list to disk.

import json
from pathlib import Path

CHECKPOINT_DIR = REPO / "cache" / "model_features"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_CHECKPOINT = (
    CHECKPOINT_DIR / "e23_retrieval_38a0b640f78b_labeled_features.parquet"
)

META_CHECKPOINT = (
    CHECKPOINT_DIR / "e23_retrieval_38a0b640f78b_training_metadata.json"
)

# Save the full 488,090 x 78 feature table, including labels.
features.to_parquet(
    FEATURE_CHECKPOINT,
    index=False,
    compression="snappy",
)

# Save everything needed to recreate the exact matrices/split.
metadata = {
    "run": "e23_retrieval_38a0b640f78b",
    "train_s1_ids": sorted(train_s1_ids),
    "val_s1_ids": sorted(val_s1_ids),
    "model_features": model_features,
    "feature_rows": int(len(features)),
    "feature_columns": int(len(features.columns)),
    "positive_pairs": int(features["label"].sum()),
}

with open(META_CHECKPOINT, "w") as f:
    json.dump(metadata, f, indent=2)

print("Feature checkpoint:")
print(FEATURE_CHECKPOINT)
print(f"Size: {FEATURE_CHECKPOINT.stat().st_size / 1024**2:.1f} MiB")

print("\nMetadata checkpoint:")
print(META_CHECKPOINT)
print(f"Size: {META_CHECKPOINT.stat().st_size / 1024:.1f} KiB")

print("\n✅ CHECKPOINT SAVED — safe to disconnect")

Feature checkpoint:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/model_features/e23_retrieval_38a0b640f78b_labeled_features.parquet
Size: 17.7 MiB

Metadata checkpoint:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/model_features/e23_retrieval_38a0b640f78b_training_metadata.json
Size: 196.6 KiB

✅ CHECKPOINT SAVED — safe to disconnect


In [1]:
# Resume Cell 1: Reload the saved E2.3 checkpoint after the server restart
# and recreate the exact train/validation matrices without recomputing features.

import json
from pathlib import Path
import pandas as pd

REPO = Path.home() / "Amazon-ML-Challenge"

CHECKPOINT_DIR = REPO / "cache" / "model_features"

FEATURE_CHECKPOINT = (
    CHECKPOINT_DIR / "e23_retrieval_38a0b640f78b_labeled_features.parquet"
)

META_CHECKPOINT = (
    CHECKPOINT_DIR / "e23_retrieval_38a0b640f78b_training_metadata.json"
)

features = pd.read_parquet(FEATURE_CHECKPOINT)

with open(META_CHECKPOINT, "r") as f:
    metadata = json.load(f)

train_s1_ids = set(metadata["train_s1_ids"])
val_s1_ids = set(metadata["val_s1_ids"])
model_features = metadata["model_features"]

train_mask = features["source1_entity_id"].isin(train_s1_ids)
val_mask = features["source1_entity_id"].isin(val_s1_ids)

X_train = features.loc[train_mask, model_features]
y_train = features.loc[train_mask, "label"]

X_val = features.loc[val_mask, model_features]
y_val = features.loc[val_mask, "label"]

print("Feature table:", features.shape)
print("Model features:", len(model_features))
print("Train pairs:", len(X_train))
print("Validation pairs:", len(X_val))
print("Train positives:", int(y_train.sum()))
print("Validation positives:", int(y_val.sum()))

print("✅ Checkpoint restored")

Feature table: (488090, 78)
Model features: 75
Train pairs: 369671
Validation pairs: 118419
Train positives: 25404
Validation positives: 8263
✅ Checkpoint restored


In [2]:
# Training Cell 1: Train the E2.3 LightGBM classifier with a live tqdm progress
# bar, validation logging, and early stopping so training progress is visible.

import lightgbm as lgb
import time
from tqdm.auto import tqdm

train_data = lgb.Dataset(
    X_train,
    label=y_train,
    feature_name=model_features,
    free_raw_data=False,
)

val_data = lgb.Dataset(
    X_val,
    label=y_val,
    reference=train_data,
    feature_name=model_features,
    free_raw_data=False,
)

params = {
    "objective": "binary",
    "metric": ["binary_logloss", "auc"],
    "learning_rate": 0.05,
    "num_leaves": 63,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "min_data_in_leaf": 50,
    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed": 42,
    "verbosity": -1,
}

NUM_BOOST_ROUND = 2000

progress_bar = tqdm(
    total=NUM_BOOST_ROUND,
    desc="LightGBM boosting",
    unit="round",
    dynamic_ncols=True,
)

last_iteration = {"value": 0}

def tqdm_callback(env):
    current_iteration = env.iteration + 1
    increment = current_iteration - last_iteration["value"]

    if increment > 0:
        progress_bar.update(increment)
        last_iteration["value"] = current_iteration

    # Show current validation metrics in the tqdm bar.
    if env.evaluation_result_list:
        metrics = {}

        for item in env.evaluation_result_list:
            dataset_name = item[0]
            metric_name = item[1]
            metric_value = item[2]

            if dataset_name == "validation":
                metrics[metric_name] = f"{metric_value:.6f}"

        if metrics:
            progress_bar.set_postfix(metrics)

# Run tqdm after metric evaluation each round.
tqdm_callback.order = 20

print(" TRAINING LIGHTGBM NOW")
print(f"Train pairs:      {len(X_train):,}")
print(f"Validation pairs: {len(X_val):,}")
print(f"Features:         {len(model_features)}")
print(f"Max rounds:       {NUM_BOOST_ROUND:,}")
print()

t0 = time.time()

try:
    model = lgb.train(
        params,
        train_data,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[train_data, val_data],
        valid_names=["train", "validation"],
        callbacks=[
            tqdm_callback,
            lgb.log_evaluation(period=25),
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=True,
            ),
        ],
    )

finally:
    progress_bar.close()

elapsed = time.time() - t0

print()
print("TRAINING COMPLETE")
print("Best iteration:", model.best_iteration)
print("Best validation scores:")
print(model.best_score["validation"])
print(f"Training time: {elapsed:.1f}s")

LightGBM boosting:   0%|          | 0/2000 [00:00<?, ?round/s]

 TRAINING LIGHTGBM NOW
Train pairs:      369,671
Validation pairs: 118,419
Features:         75
Max rounds:       2,000

Training until validation scores don't improve for 100 rounds
[25]	train's binary_logloss: 0.0506741	train's auc: 0.999066	validation's binary_logloss: 0.0515806	validation's auc: 0.998696
[50]	train's binary_logloss: 0.0228509	train's auc: 0.999487	validation's binary_logloss: 0.0241725	validation's auc: 0.999101
[75]	train's binary_logloss: 0.0143007	train's auc: 0.999657	validation's binary_logloss: 0.0163697	validation's auc: 0.999285
[100]	train's binary_logloss: 0.0108127	train's auc: 0.999782	validation's binary_logloss: 0.0137894	validation's auc: 0.999341
[125]	train's binary_logloss: 0.00887789	train's auc: 0.999852	validation's binary_logloss: 0.0127566	validation's auc: 0.999392
[150]	train's binary_logloss: 0.00756851	train's auc: 0.9999	validation's binary_logloss: 0.0122733	validation's auc: 0.999445
[175]	train's binary_logloss: 0.00662943	train's auc

In [3]:
# Cell 19: Save the trained E2.3 LightGBM model and its exact 75-feature schema
# so the model can be restored without retraining after a server restart.

import json
from pathlib import Path

MODEL_DIR = REPO / "cache" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = (
    MODEL_DIR / "e23_lgbm_38a0b640f78b.txt"
)

MODEL_META_PATH = (
    MODEL_DIR / "e23_lgbm_38a0b640f78b_metadata.json"
)

model.save_model(
    str(MODEL_PATH),
    num_iteration=model.best_iteration,
)

model_metadata = {
    "run": "e23_retrieval_38a0b640f78b",
    "model_type": "LightGBM",
    "best_iteration": int(model.best_iteration),
    "model_features": model_features,
    "train_pairs": int(len(X_train)),
    "validation_pairs": int(len(X_val)),
    "train_positives": int(y_train.sum()),
    "validation_positives": int(y_val.sum()),
    "validation_binary_logloss": float(
        model.best_score["validation"]["binary_logloss"]
    ),
    "validation_auc": float(
        model.best_score["validation"]["auc"]
    ),
}

with open(MODEL_META_PATH, "w") as f:
    json.dump(model_metadata, f, indent=2)

print("Model saved:")
print(MODEL_PATH)
print(f"Size: {MODEL_PATH.stat().st_size / 1024:.1f} KiB")

print("\nMetadata saved:")
print(MODEL_META_PATH)

print("\n✅ TRAINED LIGHTGBM CHECKPOINTED")

Model saved:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/models/e23_lgbm_38a0b640f78b.txt
Size: 1772.3 KiB

Metadata saved:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/models/e23_lgbm_38a0b640f78b_metadata.json

✅ TRAINED LIGHTGBM CHECKPOINTED


In [4]:
# Cell 20: Score all validation candidate pairs with the trained LightGBM model,
# show prediction progress with tqdm, and save the probabilities for threshold tuning.

import numpy as np
from tqdm.auto import tqdm

PRED_CHUNK_SIZE = 25_000

val_scores_parts = []

for start in tqdm(
    range(0, len(X_val), PRED_CHUNK_SIZE),
    desc="Scoring validation pairs",
    unit="chunk",
    dynamic_ncols=True,
):
    end = min(start + PRED_CHUNK_SIZE, len(X_val))

    chunk_scores = model.predict(
        X_val.iloc[start:end],
        num_iteration=model.best_iteration,
    )

    val_scores_parts.append(chunk_scores)

val_scores = np.concatenate(val_scores_parts)

# Attach probabilities back to the validation candidate rows.
val_results = features.loc[
    val_mask,
    ["source1_entity_id", "candidate_entity_id", "label"]
].copy()

val_results["score"] = val_scores.astype("float32")

PRED_PATH = (
    REPO
    / "cache"
    / "predictions"
    / "e23_lgbm_38a0b640f78b_validation.parquet"
)

PRED_PATH.parent.mkdir(parents=True, exist_ok=True)

val_results.to_parquet(
    PRED_PATH,
    index=False,
    compression="snappy",
)

print("Validation predictions:", val_results.shape)
print("Positive labels:", int(val_results["label"].sum()))
print("Score min:", float(val_results["score"].min()))
print("Score max:", float(val_results["score"].max()))
print("Score mean:", float(val_results["score"].mean()))
print("\nSaved to:")
print(PRED_PATH)

print("\n✅ Validation probabilities generated and checkpointed")

Scoring validation pairs:   0%|          | 0/5 [00:00<?, ?chunk/s]

Validation predictions: (118419, 4)
Positive labels: 8263
Score min: 2.2736253413313534e-06
Score max: 0.9999555349349976
Score mean: 0.06967224180698395

Saved to:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/predictions/e23_lgbm_38a0b640f78b_validation.parquet

✅ Validation probabilities generated and checkpointed


In [5]:
# Cell 21: Sweep decision thresholds using the true competition macro-F0.5 metric.
# This includes retrieval-missed ground-truth links, not just labels present in candidates.

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

BETA = 0.5
BETA2 = BETA ** 2

# Reload the complete training ground truth.
GT_PATH = (
    REPO
    / "student_resource"
    / "dataset"
    / "train"
    / "train_ground_truth.tsv"
)

gt = pd.read_csv(GT_PATH, sep="\t")

# Keep only the exact validation Source1 entities.
val_gt = gt[
    gt["source1_entity_id"].isin(val_s1_ids)
].copy()

# Count total TRUE links for every validation entity.
# This deliberately includes links that retrieval failed to generate as candidates.
val_gt["truth_count"] = (
    val_gt["matched_entity_ids"]
    .fillna("")
    .astype(str)
    .apply(lambda x: 0 if x.strip() == "" else len(x.split(",")))
)

truth_count_map = dict(
    zip(val_gt["source1_entity_id"], val_gt["truth_count"])
)

# Encode validation entities as integer indices for very fast bincount operations.
val_entities = sorted(val_s1_ids)
entity_to_idx = {
    entity_id: i
    for i, entity_id in enumerate(val_entities)
}

entity_idx = (
    val_results["source1_entity_id"]
    .map(entity_to_idx)
    .to_numpy(dtype=np.int32)
)

scores = val_results["score"].to_numpy(dtype=np.float32)
labels = val_results["label"].to_numpy(dtype=np.uint8)

total_truth = np.array(
    [truth_count_map.get(entity_id, 0) for entity_id in val_entities],
    dtype=np.int32,
)

n_entities = len(val_entities)

# Dense threshold sweep.
thresholds = np.linspace(0.01, 0.995, 500)

results = []

for threshold in tqdm(
    thresholds,
    desc="Sweeping F0.5 thresholds",
    unit="threshold",
    dynamic_ncols=True,
):
    predicted = scores >= threshold

    pred_count = np.bincount(
        entity_idx[predicted],
        minlength=n_entities,
    )

    tp = np.bincount(
        entity_idx[predicted & (labels == 1)],
        minlength=n_entities,
    )

    fp = pred_count - tp
    fn = total_truth - tp

    denominator = (
        (1 + BETA2) * tp
        + BETA2 * fn
        + fp
    )

    # Official special case:
    # empty ground truth + empty prediction => score 1.
    entity_f05 = np.zeros(n_entities, dtype=np.float64)

    both_empty = (total_truth == 0) & (pred_count == 0)
    entity_f05[both_empty] = 1.0

    normal = denominator > 0

    entity_f05[normal] = (
        (1 + BETA2) * tp[normal]
        / denominator[normal]
    )

    results.append(
        (
            float(threshold),
            float(entity_f05.mean()),
            int(tp.sum()),
            int(fp.sum()),
            int(fn.sum()),
        )
    )

threshold_results = pd.DataFrame(
    results,
    columns=[
        "threshold",
        "macro_f0.5",
        "tp",
        "fp",
        "fn",
    ],
)

best_row = threshold_results.loc[
    threshold_results["macro_f0.5"].idxmax()
]

print("\n🏆 BEST VALIDATION THRESHOLD")
print(f"Threshold:  {best_row['threshold']:.6f}")
print(f"Macro F0.5: {best_row['macro_f0.5']:.6f}")
print(f"TP:         {int(best_row['tp']):,}")
print(f"FP:         {int(best_row['fp']):,}")
print(f"FN:         {int(best_row['fn']):,}")

print("\nTop 10 thresholds:")
display(
    threshold_results
    .sort_values("macro_f0.5", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

Sweeping F0.5 thresholds:   0%|          | 0/500 [00:00<?, ?threshold/s]


🏆 BEST VALIDATION THRESHOLD
Threshold:  0.710752
Macro F0.5: 0.962594
TP:         7,887
FP:         143
FN:         582

Top 10 thresholds:


,threshold,macro_f0.5,tp,fp,fn
0,0.710752,0.962594,7887,143,582
1,0.714699,0.962529,7884,142,585
2,0.742335,0.962514,7854,131,615
3,0.722595,0.962508,7876,140,593
4,0.712725,0.962480,7884,143,585
5,0.623898,0.962469,7944,168,525
6,0.616002,0.962460,7950,171,519
7,0.716673,0.962454,7882,142,587
8,0.720621,0.962448,7878,141,591
9,0.612054,0.962442,7951,172,518


In [6]:
# Cell 22: Evaluate the best E2.3 threshold separately for India and the US
# to identify where the remaining validation errors and F0.5 loss come from.

BEST_THRESHOLD = float(best_row["threshold"])

# Reload the normalized 10k Source1 table to obtain each entity's country.
SOURCE1_PATH = (
    REPO
    / "cache"
    / "normalized"
    / "e21_lgbm_c449c0c6b281_source1.parquet"
)

source1_eval = pd.read_parquet(
    SOURCE1_PATH,
    columns=["entity_id", "country"],
)

country_map = dict(
    zip(source1_eval["entity_id"], source1_eval["country"])
)

entity_country = np.array(
    [country_map[eid] for eid in val_entities]
)

predicted = scores >= BEST_THRESHOLD

pred_count = np.bincount(
    entity_idx[predicted],
    minlength=n_entities,
)

tp = np.bincount(
    entity_idx[predicted & (labels == 1)],
    minlength=n_entities,
)

fp = pred_count - tp
fn = total_truth - tp

denominator = (
    (1 + BETA2) * tp
    + BETA2 * fn
    + fp
)

entity_f05 = np.zeros(n_entities, dtype=np.float64)

both_empty = (total_truth == 0) & (pred_count == 0)
entity_f05[both_empty] = 1.0

normal = denominator > 0
entity_f05[normal] = (
    (1 + BETA2) * tp[normal]
    / denominator[normal]
)

rows = []

for country in sorted(set(entity_country)):
    mask = entity_country == country

    rows.append({
        "country": country,
        "entities": int(mask.sum()),
        "macro_f0.5": float(entity_f05[mask].mean()),
        "tp": int(tp[mask].sum()),
        "fp": int(fp[mask].sum()),
        "fn": int(fn[mask].sum()),
        "truth_links": int(total_truth[mask].sum()),
    })

country_results = pd.DataFrame(rows)

print(f"Threshold: {BEST_THRESHOLD:.6f}")
display(country_results)

print(
    "\nOverall macro F0.5:",
    f"{entity_f05.mean():.6f}"
)

Threshold: 0.710752


,country,entities,macro_f0.5,tp,fp,fn,truth_links
0,India,953,0.945960,2979,70,325,3304
1,US,1479,0.973312,4908,73,257,5165



Overall macro F0.5: 0.962594


In [7]:
# Cell 23: Decompose validation false negatives into retrieval misses versus
# classifier rejections, separately for India and the US.

BEST_THRESHOLD = float(best_row["threshold"])

# True candidate links actually recovered by retrieval.
recovered_true = val_results[val_results["label"] == 1].copy()

# Of the recovered true links, these were rejected by LightGBM.
classifier_rejected = recovered_true[
    recovered_true["score"] < BEST_THRESHOLD
].copy()

# Count recovered true links by Source1 entity.
recovered_true_count = (
    recovered_true
    .groupby("source1_entity_id")
    .size()
    .to_dict()
)

# Count classifier-rejected true links by Source1 entity.
classifier_rejected_count = (
    classifier_rejected
    .groupby("source1_entity_id")
    .size()
    .to_dict()
)

error_rows = []

for entity_id in val_entities:
    country = country_map[entity_id]

    truth = int(truth_count_map.get(entity_id, 0))
    recovered = int(recovered_true_count.get(entity_id, 0))
    rejected = int(classifier_rejected_count.get(entity_id, 0))

    retrieval_missed = truth - recovered

    error_rows.append({
        "source1_entity_id": entity_id,
        "country": country,
        "truth_links": truth,
        "retrieved_true_links": recovered,
        "retrieval_misses": retrieval_missed,
        "classifier_rejections": rejected,
    })

error_breakdown = pd.DataFrame(error_rows)

country_error_summary = (
    error_breakdown
    .groupby("country", as_index=False)
    .agg(
        entities=("source1_entity_id", "count"),
        truth_links=("truth_links", "sum"),
        retrieved_true_links=("retrieved_true_links", "sum"),
        retrieval_misses=("retrieval_misses", "sum"),
        classifier_rejections=("classifier_rejections", "sum"),
    )
)

country_error_summary["retrieval_miss_rate"] = (
    country_error_summary["retrieval_misses"]
    / country_error_summary["truth_links"]
)

country_error_summary["classifier_reject_rate_of_retrieved"] = (
    country_error_summary["classifier_rejections"]
    / country_error_summary["retrieved_true_links"]
)

display(country_error_summary)

print("\nTotal FN decomposition:")
print(
    "Retrieval misses:",
    int(country_error_summary["retrieval_misses"].sum())
)
print(
    "Classifier rejections:",
    int(country_error_summary["classifier_rejections"].sum())
)

,country,entities,truth_links,retrieved_true_links,retrieval_misses,classifier_rejections,retrieval_miss_rate,classifier_reject_rate_of_retrieved
0,India,953,3304,3164,140,185,0.042373,0.058470
1,US,1479,5165,5099,66,191,0.012778,0.037458



Total FN decomposition:
Retrieval misses: 206
Classifier rejections: 376


In [8]:
# Cell 24: Sweep separate LightGBM decision thresholds for India and the US
# to test whether country-specific thresholds improve macro-F0.5 over one global threshold.

country_threshold_results = {}

for country in ["India", "US"]:
    country_entity_mask = entity_country == country
    country_entity_indices = np.where(country_entity_mask)[0]

    # Map global entity indices -> compact country-local indices.
    global_to_local = {
        global_idx: local_idx
        for local_idx, global_idx in enumerate(country_entity_indices)
    }

    pair_mask = np.isin(entity_idx, country_entity_indices)

    country_scores = scores[pair_mask]
    country_labels = labels[pair_mask]
    country_global_idx = entity_idx[pair_mask]

    country_local_idx = np.array(
        [global_to_local[i] for i in country_global_idx],
        dtype=np.int32,
    )

    country_truth = total_truth[country_entity_indices]
    n_country_entities = len(country_entity_indices)

    rows = []

    for threshold in tqdm(
        thresholds,
        desc=f"Sweeping {country}",
        unit="threshold",
        dynamic_ncols=True,
    ):
        predicted = country_scores >= threshold

        pred_count = np.bincount(
            country_local_idx[predicted],
            minlength=n_country_entities,
        )

        tp_country = np.bincount(
            country_local_idx[predicted & (country_labels == 1)],
            minlength=n_country_entities,
        )

        fp_country = pred_count - tp_country
        fn_country = country_truth - tp_country

        denominator = (
            (1 + BETA2) * tp_country
            + BETA2 * fn_country
            + fp_country
        )

        entity_scores = np.zeros(
            n_country_entities,
            dtype=np.float64,
        )

        both_empty = (
            (country_truth == 0)
            & (pred_count == 0)
        )
        entity_scores[both_empty] = 1.0

        normal = denominator > 0
        entity_scores[normal] = (
            (1 + BETA2) * tp_country[normal]
            / denominator[normal]
        )

        rows.append({
            "threshold": float(threshold),
            "macro_f0.5": float(entity_scores.mean()),
            "tp": int(tp_country.sum()),
            "fp": int(fp_country.sum()),
            "fn": int(fn_country.sum()),
        })

    result = pd.DataFrame(rows)
    country_threshold_results[country] = result

    best = result.loc[result["macro_f0.5"].idxmax()]

    print(f"\n🏆 {country}")
    print(f"Best threshold: {best['threshold']:.6f}")
    print(f"Macro F0.5:     {best['macro_f0.5']:.6f}")
    print(f"TP:             {int(best['tp']):,}")
    print(f"FP:             {int(best['fp']):,}")
    print(f"FN:             {int(best['fn']):,}")

    display(
        result.sort_values(
            "macro_f0.5",
            ascending=False,
        ).head(5)
    )

Sweeping India:   0%|          | 0/500 [00:00<?, ?threshold/s]


🏆 India
Best threshold: 0.616002
Macro F0.5:     0.946879
TP:             3,012
FP:             84
FN:             292


,threshold,macro_f0.5,tp,fp,fn
307,0.616002,0.946879,3012,84,292
304,0.610080,0.946833,3013,85,291
308,0.617976,0.946760,3011,84,293
310,0.621924,0.946705,3010,83,294
311,0.623898,0.946705,3010,83,294


Sweeping US:   0%|          | 0/500 [00:00<?, ?threshold/s]


🏆 US
Best threshold: 0.742335
Macro F0.5:     0.973603
TP:             4,885
FP:             65
FN:             280


,threshold,macro_f0.5,tp,fp,fn
371,0.742335,0.973603,4885,65,280
368,0.736413,0.973589,4889,68,276
367,0.734439,0.973538,4891,70,274
369,0.738387,0.973387,4887,68,278
370,0.740361,0.973387,4887,68,278


In [9]:
# Cell 25: Apply the best country-specific thresholds simultaneously and compute
# the combined validation macro-F0.5 to see the gain over the global threshold.

COUNTRY_THRESHOLDS = {
    "India": 0.616002,
    "US": 0.742335,
}

pair_countries = np.array(
    [country_map[eid] for eid in val_results["source1_entity_id"]]
)

pair_thresholds = np.array(
    [COUNTRY_THRESHOLDS[c] for c in pair_countries],
    dtype=np.float32,
)

predicted = scores >= pair_thresholds

pred_count = np.bincount(
    entity_idx[predicted],
    minlength=n_entities,
)

tp_country_specific = np.bincount(
    entity_idx[predicted & (labels == 1)],
    minlength=n_entities,
)

fp_country_specific = pred_count - tp_country_specific
fn_country_specific = total_truth - tp_country_specific

denominator = (
    (1 + BETA2) * tp_country_specific
    + BETA2 * fn_country_specific
    + fp_country_specific
)

entity_f05_country_specific = np.zeros(
    n_entities,
    dtype=np.float64,
)

both_empty = (
    (total_truth == 0)
    & (pred_count == 0)
)

entity_f05_country_specific[both_empty] = 1.0

normal = denominator > 0

entity_f05_country_specific[normal] = (
    (1 + BETA2) * tp_country_specific[normal]
    / denominator[normal]
)

combined_score = entity_f05_country_specific.mean()

print("Country-specific thresholds:")
print(COUNTRY_THRESHOLDS)

print("\nCombined validation results:")
print(f"Macro F0.5: {combined_score:.6f}")
print(f"TP:         {int(tp_country_specific.sum()):,}")
print(f"FP:         {int(fp_country_specific.sum()):,}")
print(f"FN:         {int(fn_country_specific.sum()):,}")

print("\nComparison:")
print(f"Global threshold score:   {best_row['macro_f0.5']:.6f}")
print(f"Country-specific score:   {combined_score:.6f}")
print(
    f"Absolute improvement:     "
    f"{combined_score - best_row['macro_f0.5']:+.6f}"
)

Country-specific thresholds:
{'India': 0.616002, 'US': 0.742335}

Combined validation results:
Macro F0.5: 0.963131
TP:         7,897
FP:         149
FN:         572

Comparison:
Global threshold score:   0.962594
Country-specific score:   0.963131
Absolute improvement:     +0.000537


In [10]:
# Cell 26: Save the selected country-specific thresholds and validation metrics
# so inference can reproduce the exact decoder without re-running threshold sweeps.

import json

DECODER_PATH = (
    REPO
    / "cache"
    / "models"
    / "e23_lgbm_38a0b640f78b_decoder.json"
)

decoder_config = {
    "model_run": "e23_lgbm_38a0b640f78b",
    "metric": "macro_f0.5",
    "beta": 0.5,

    "global_threshold": float(best_row["threshold"]),
    "global_validation_macro_f0.5": float(
        best_row["macro_f0.5"]
    ),

    "country_thresholds": {
        "India": 0.616002,
        "US": 0.742335,
    },

    "country_specific_validation_macro_f0.5": float(
        combined_score
    ),

    "validation_tp": int(tp_country_specific.sum()),
    "validation_fp": int(fp_country_specific.sum()),
    "validation_fn": int(fn_country_specific.sum()),

    # France is unseen in training; this is intentionally not finalized yet.
    "france_threshold": None,
}

with open(DECODER_PATH, "w") as f:
    json.dump(decoder_config, f, indent=2)

print("Decoder config saved:")
print(DECODER_PATH)

print("\nSaved thresholds:")
print("India:", decoder_config["country_thresholds"]["India"])
print("US:   ", decoder_config["country_thresholds"]["US"])

print(
    "\nValidation macro F0.5:",
    f"{decoder_config['country_specific_validation_macro_f0.5']:.6f}",
)

print("\n E2.3 decoder checkpointed")

Decoder config saved:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/models/e23_lgbm_38a0b640f78b_decoder.json

Saved thresholds:
India: 0.616002
US:    0.742335

Validation macro F0.5: 0.963131

 E2.3 decoder checkpointed


In [11]:
# Cell 27: Inspect LightGBM feature importance to see which legacy and E2.3
# address-retrieval features contribute most to the trained classifier.

importance = pd.DataFrame({
    "feature": model.feature_name(),
    "gain": model.feature_importance(
        importance_type="gain",
        iteration=model.best_iteration,
    ),
    "split": model.feature_importance(
        importance_type="split",
        iteration=model.best_iteration,
    ),
})

importance["gain_pct"] = (
    100 * importance["gain"] / importance["gain"].sum()
)

importance = importance.sort_values(
    "gain",
    ascending=False,
).reset_index(drop=True)

print("Top 25 features by gain:")
display(
    importance[
        ["feature", "gain_pct", "split"]
    ].head(25)
)

e23_feature_set = set(E23_MODEL_FEATURES)

e23_importance = importance[
    importance["feature"].isin(e23_feature_set)
].copy()

print(
    "\nTotal gain from E2.3-specific features:",
    f"{e23_importance['gain_pct'].sum():.2f}%"
)

print("\nE2.3-specific feature importance:")
display(
    e23_importance[
        ["feature", "gain_pct", "split"]
    ].reset_index(drop=True)
)

Top 25 features by gain:


,feature,gain_pct,split
0,address_tfidf_similarity,43.014107,560
1,retrieval_channel_count,11.959685,37
2,relative_address_tfidf_rank,7.637506,318
3,fusion_score,7.480720,493
4,address_token_jaccard,5.500558,872
5,name_character_similarity,5.334405,1241
6,name_token_jaccard,3.221939,916
7,address_length_ratio,2.145782,788
8,numeric_token_conflict,1.933664,232
9,numeric_token_overlap,1.914968,401


NameError: name 'E23_MODEL_FEATURES' is not defined

In [12]:
# Cell 27B: Recreate the E2.3-specific feature list after the kernel restart
# and measure how much total LightGBM gain comes from the new retrieval features.

E23_MODEL_FEATURES = [
    "address_tfidf_similarity",
    "address_tfidf_rank",
    "address_route_postcode",
    "address_route_postcode_prefix",
    "address_route_numeric",
    "address_route_rare_token",
    "address_route_country_fallback",
    "address_route_count",
    "address_preblock_pool_size",
    "has_address_tfidf",
    "block_address_tfidf",
    "is_address_tfidf_only",
    "retrieval_channel_count",
    "relative_address_tfidf_rank",
    "address_tfidf_gap_to_best",
]

e23_importance = importance[
    importance["feature"].isin(E23_MODEL_FEATURES)
].copy()

print(
    "Total gain from E2.3-specific features:",
    f"{e23_importance['gain_pct'].sum():.2f}%"
)

display(
    e23_importance[
        ["feature", "gain_pct", "split"]
    ].reset_index(drop=True)
)

Total gain from E2.3-specific features: 63.51%


,feature,gain_pct,split
0,address_tfidf_similarity,43.014107,560
1,retrieval_channel_count,11.959685,37
2,relative_address_tfidf_rank,7.637506,318
3,address_route_count,0.316272,88
4,address_tfidf_gap_to_best,0.263988,597
5,address_preblock_pool_size,0.214910,481
6,address_tfidf_rank,0.068240,119
7,address_route_rare_token,0.014583,25
8,address_route_numeric,0.010419,27
9,address_route_postcode,0.006917,21


In [13]:
# Cell 28: Save LightGBM feature importance and the E2.3 gain summary so the
# experiment record survives notebook/server restarts and can be compared later.

IMPORTANCE_PATH = (
    REPO
    / "experiments"
    / "e23_lgbm_38a0b640f78b_feature_importance.csv"
)

SUMMARY_PATH = (
    REPO
    / "experiments"
    / "e23_lgbm_38a0b640f78b_summary.json"
)

IMPORTANCE_PATH.parent.mkdir(parents=True, exist_ok=True)

importance.to_csv(
    IMPORTANCE_PATH,
    index=False,
)

experiment_summary = {
    "run": "e23_lgbm_38a0b640f78b",
    "best_iteration": int(model.best_iteration),
    "validation_auc": float(
        model.best_score["validation"]["auc"]
    ),
    "validation_logloss": float(
        model.best_score["validation"]["binary_logloss"]
    ),
    "global_threshold": float(best_row["threshold"]),
    "global_macro_f0.5": float(best_row["macro_f0.5"]),
    "country_thresholds": {
        "India": 0.616002,
        "US": 0.742335,
    },
    "country_specific_macro_f0.5": float(combined_score),
    "e23_feature_gain_pct": float(
        e23_importance["gain_pct"].sum()
    ),
    "top_feature": str(importance.iloc[0]["feature"]),
    "top_feature_gain_pct": float(
        importance.iloc[0]["gain_pct"]
    ),
}

with open(SUMMARY_PATH, "w") as f:
    json.dump(
        experiment_summary,
        f,
        indent=2,
    )

print("Feature importance saved:")
print(IMPORTANCE_PATH)

print("\nExperiment summary saved:")
print(SUMMARY_PATH)

print("\nKey result:")
print(
    f"E2.3-specific feature gain: "
    f"{experiment_summary['e23_feature_gain_pct']:.2f}%"
)
print(
    f"Best validation macro F0.5: "
    f"{experiment_summary['country_specific_macro_f0.5']:.6f}"
)

print("\n✅ E2.3 model evaluation checkpointed")

Feature importance saved:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/experiments/e23_lgbm_38a0b640f78b_feature_importance.csv

Experiment summary saved:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/experiments/e23_lgbm_38a0b640f78b_summary.json

Key result:
E2.3-specific feature gain: 63.51%
Best validation macro F0.5: 0.963131

✅ E2.3 model evaluation checkpointed


## E2.4 — XGBoost benchmark / ensemble candidate


In [16]:
# XGBoost Cell 1: Check whether XGBoost is installed and report its version
# before we prepare the second tree-based model.

try:
    import xgboost as xgb
    print("XGBoost version:", xgb.__version__)
    print("✅ XGBoost is available")
except Exception as e:
    print("❌ XGBoost import failed")
    print(type(e).__name__ + ":", e)

XGBoost version: 2.1.4
✅ XGBoost is available


In [17]:
# XGBoost Cell 2: Build memory-efficient QuantileDMatrix objects from the exact
# same train/validation pairs used by LightGBM. This prepares XGBoost only;
# model training has NOT started yet.

import xgboost as xgb
import time

t0 = time.time()

dtrain = xgb.QuantileDMatrix(
    X_train,
    y_train,
    max_bin=256,
    feature_names=model_features,
)

dval = xgb.QuantileDMatrix(
    X_val,
    y_val,
    max_bin=256,
    feature_names=model_features,
    ref=dtrain,
)

elapsed = time.time() - t0

print(f"XGBoost matrices built in {elapsed:.1f}s")
print("Train rows:", dtrain.num_row())
print("Validation rows:", dval.num_row())
print("Features:", dtrain.num_col())

assert dtrain.num_row() == 369671
assert dval.num_row() == 118419
assert dtrain.num_col() == 75

print("✅ XGBoost matrices ready")

XGBoost matrices built in 0.6s
Train rows: 369671
Validation rows: 118419
Features: 75
✅ XGBoost matrices ready


In [18]:
# XGBoost Cell 3: Train the second tabular model on the V100 GPU using the
# exact same 75 features and entity-level split as LightGBM, with live progress.

import time
from tqdm.auto import tqdm
import xgboost as xgb

NUM_BOOST_ROUND = 2000
EARLY_STOPPING_ROUNDS = 100

params_xgb = {
    "objective": "binary:logistic",
    "eval_metric": ["logloss", "auc"],
    "learning_rate": 0.05,
    "max_depth": 8,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "max_bin": 256,
    "tree_method": "hist",
    "device": "cuda",
    "seed": 42,
}

progress = tqdm(
    total=NUM_BOOST_ROUND,
    desc="XGBoost boosting",
    unit="round",
    dynamic_ncols=True,
)

class TqdmXGBCallback(xgb.callback.TrainingCallback):
    def __init__(self, bar):
        self.bar = bar
        self.last_iter = 0

    def after_iteration(self, model, epoch, evals_log):
        current_iter = epoch + 1
        increment = current_iter - self.last_iter

        if increment > 0:
            self.bar.update(increment)
            self.last_iter = current_iter

        postfix = {}

        if "validation" in evals_log:
            if "logloss" in evals_log["validation"]:
                postfix["logloss"] = f"{evals_log['validation']['logloss'][-1]:.6f}"
            if "auc" in evals_log["validation"]:
                postfix["auc"] = f"{evals_log['validation']['auc'][-1]:.6f}"

        if postfix:
            self.bar.set_postfix(postfix)

        return False


print("🚀 TRAINING XGBOOST NOW")
print(f"Train pairs:      {dtrain.num_row():,}")
print(f"Validation pairs: {dval.num_row():,}")
print(f"Features:         {dtrain.num_col()}")
print(f"Max rounds:       {NUM_BOOST_ROUND:,}")
print("Device:           CUDA / V100")
print()

evals_result = {}

t0 = time.time()

try:
    xgb_model = xgb.train(
        params=params_xgb,
        dtrain=dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[
            (dtrain, "train"),
            (dval, "validation"),
        ],
        evals_result=evals_result,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose_eval=25,
        callbacks=[
            TqdmXGBCallback(progress),
        ],
    )

finally:
    progress.close()

elapsed = time.time() - t0

print()
print("✅ XGBOOST TRAINING COMPLETE")
print("Best iteration:", xgb_model.best_iteration)
print("Best score:", xgb_model.best_score)
print(f"Training time: {elapsed:.1f}s")

XGBoost boosting:   0%|          | 0/2000 [00:00<?, ?round/s]

🚀 TRAINING XGBOOST NOW
Train pairs:      369,671
Validation pairs: 118,419
Features:         75
Max rounds:       2,000
Device:           CUDA / V100

[0]	train-logloss:0.25849	train-auc:0.98758	validation-logloss:0.26005	validation-auc:0.98820
[25]	train-logloss:0.06882	train-auc:0.99907	validation-logloss:0.06971	validation-auc:0.99869
[50]	train-logloss:0.02793	train-auc:0.99948	validation-logloss:0.02943	validation-auc:0.99913
[75]	train-logloss:0.01566	train-auc:0.99966	validation-logloss:0.01782	validation-auc:0.99931
[100]	train-logloss:0.01150	train-auc:0.99975	validation-logloss:0.01428	validation-auc:0.99937
[125]	train-logloss:0.00970	train-auc:0.99980	validation-logloss:0.01308	validation-auc:0.99940
[150]	train-logloss:0.00863	train-auc:0.99984	validation-logloss:0.01250	validation-auc:0.99944
[175]	train-logloss:0.00788	train-auc:0.99987	validation-logloss:0.01218	validation-auc:0.99947
[200]	train-logloss:0.00738	train-auc:0.99988	validation-logloss:0.01204	validation-au

In [19]:
# XGBoost Cell 4: Save the trained XGBoost model and its exact training metadata
# before threshold tuning so the model survives any notebook/server restart.

import json
from pathlib import Path

XGB_MODEL_DIR = REPO / "cache" / "models"
XGB_MODEL_DIR.mkdir(parents=True, exist_ok=True)

XGB_MODEL_PATH = (
    XGB_MODEL_DIR / "e24_xgb_38a0b640f78b.json"
)

XGB_META_PATH = (
    XGB_MODEL_DIR / "e24_xgb_38a0b640f78b_metadata.json"
)

xgb_model.save_model(XGB_MODEL_PATH)

xgb_metadata = {
    "run": "e24_xgb_38a0b640f78b",
    "base_retrieval_run": "e23_retrieval_38a0b640f78b",
    "model_type": "XGBoost",
    "xgboost_version": xgb.__version__,
    "best_iteration": int(xgb_model.best_iteration),
    "best_score": float(xgb_model.best_score),
    "model_features": model_features,
    "train_pairs": int(len(X_train)),
    "validation_pairs": int(len(X_val)),
    "train_positives": int(y_train.sum()),
    "validation_positives": int(y_val.sum()),
    "device": "cuda",
}

with open(XGB_META_PATH, "w") as f:
    json.dump(xgb_metadata, f, indent=2)

print("XGBoost model saved:")
print(XGB_MODEL_PATH)
print(f"Size: {XGB_MODEL_PATH.stat().st_size / 1024:.1f} KiB")

print("\nMetadata saved:")
print(XGB_META_PATH)

print("\n✅ XGBoost checkpointed")

XGBoost model saved:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/models/e24_xgb_38a0b640f78b.json
Size: 4432.2 KiB

Metadata saved:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/models/e24_xgb_38a0b640f78b_metadata.json

✅ XGBoost checkpointed


In [20]:
# XGBoost Cell 5: Score the validation set with the trained XGBoost model,
# show progress, and save probabilities for threshold and ensemble analysis.

import numpy as np
from tqdm.auto import tqdm

XGB_PRED_CHUNK_SIZE = 25_000

xgb_val_scores_parts = []

for start in tqdm(
    range(0, len(X_val), XGB_PRED_CHUNK_SIZE),
    desc="Scoring XGBoost validation pairs",
    unit="chunk",
    dynamic_ncols=True,
):
    end = min(start + XGB_PRED_CHUNK_SIZE, len(X_val))

    chunk_matrix = xgb.DMatrix(
        X_val.iloc[start:end],
        feature_names=model_features,
    )

    chunk_scores = xgb_model.predict(
        chunk_matrix,
        iteration_range=(0, xgb_model.best_iteration + 1),
    )

    xgb_val_scores_parts.append(chunk_scores)

xgb_val_scores = np.concatenate(xgb_val_scores_parts)

xgb_val_results = features.loc[
    val_mask,
    ["source1_entity_id", "candidate_entity_id", "label"]
].copy()

xgb_val_results["score"] = xgb_val_scores.astype("float32")

XGB_PRED_PATH = (
    REPO
    / "cache"
    / "predictions"
    / "e24_xgb_38a0b640f78b_validation.parquet"
)

XGB_PRED_PATH.parent.mkdir(parents=True, exist_ok=True)

xgb_val_results.to_parquet(
    XGB_PRED_PATH,
    index=False,
    compression="snappy",
)

print("Validation predictions:", xgb_val_results.shape)
print("Positive labels:", int(xgb_val_results["label"].sum()))
print("Score min:", float(xgb_val_results["score"].min()))
print("Score max:", float(xgb_val_results["score"].max()))
print("Score mean:", float(xgb_val_results["score"].mean()))

print("\nSaved to:")
print(XGB_PRED_PATH)

print("\n✅ XGBoost validation probabilities generated")

Scoring XGBoost validation pairs:   0%|          | 0/5 [00:00<?, ?chunk/s]

Validation predictions: (118419, 4)
Positive labels: 8263
Score min: 1.2876455457444536e-07
Score max: 0.9999918937683105
Score mean: 0.06974166631698608

Saved to:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/predictions/e24_xgb_38a0b640f78b_validation.parquet

✅ XGBoost validation probabilities generated


In [21]:
# XGBoost Cell 6: Sweep XGBoost thresholds using the exact competition macro-F0.5
# metric, including retrieval-missed true links, for direct comparison with LightGBM.

xgb_scores = xgb_val_results["score"].to_numpy(dtype=np.float32)
xgb_labels = xgb_val_results["label"].to_numpy(dtype=np.uint8)

xgb_threshold_results = []

for threshold in tqdm(
    thresholds,
    desc="Sweeping XGBoost F0.5 thresholds",
    unit="threshold",
    dynamic_ncols=True,
):
    predicted = xgb_scores >= threshold

    pred_count = np.bincount(
        entity_idx[predicted],
        minlength=n_entities,
    )

    tp_xgb = np.bincount(
        entity_idx[predicted & (xgb_labels == 1)],
        minlength=n_entities,
    )

    fp_xgb = pred_count - tp_xgb
    fn_xgb = total_truth - tp_xgb

    denominator = (
        (1 + BETA2) * tp_xgb
        + BETA2 * fn_xgb
        + fp_xgb
    )

    entity_f05_xgb = np.zeros(
        n_entities,
        dtype=np.float64,
    )

    both_empty = (
        (total_truth == 0)
        & (pred_count == 0)
    )

    entity_f05_xgb[both_empty] = 1.0

    normal = denominator > 0

    entity_f05_xgb[normal] = (
        (1 + BETA2) * tp_xgb[normal]
        / denominator[normal]
    )

    xgb_threshold_results.append(
        (
            float(threshold),
            float(entity_f05_xgb.mean()),
            int(tp_xgb.sum()),
            int(fp_xgb.sum()),
            int(fn_xgb.sum()),
        )
    )

xgb_threshold_results = pd.DataFrame(
    xgb_threshold_results,
    columns=[
        "threshold",
        "macro_f0.5",
        "tp",
        "fp",
        "fn",
    ],
)

xgb_best_row = xgb_threshold_results.loc[
    xgb_threshold_results["macro_f0.5"].idxmax()
]

print("\n🏆 BEST XGBOOST VALIDATION THRESHOLD")
print(f"Threshold:  {xgb_best_row['threshold']:.6f}")
print(f"Macro F0.5: {xgb_best_row['macro_f0.5']:.6f}")
print(f"TP:         {int(xgb_best_row['tp']):,}")
print(f"FP:         {int(xgb_best_row['fp']):,}")
print(f"FN:         {int(xgb_best_row['fn']):,}")

print("\nHead-to-head:")
print(f"LightGBM global: {best_row['macro_f0.5']:.6f}")
print(f"XGBoost global:  {xgb_best_row['macro_f0.5']:.6f}")
print(
    "Difference:      "
    f"{xgb_best_row['macro_f0.5'] - best_row['macro_f0.5']:+.6f}"
)

print("\nTop 10 XGBoost thresholds:")
display(
    xgb_threshold_results
    .sort_values("macro_f0.5", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

Sweeping XGBoost F0.5 thresholds:   0%|          | 0/500 [00:00<?, ?threshold/s]


🏆 BEST XGBOOST VALIDATION THRESHOLD
Threshold:  0.687064
Macro F0.5: 0.963158
TP:         7,918
FP:         155
FN:         551

Head-to-head:
LightGBM global: 0.962594
XGBoost global:  0.963158
Difference:      +0.000564

Top 10 XGBoost thresholds:


,threshold,macro_f0.5,tp,fp,fn
0,0.687064,0.963158,7918,155,551
1,0.689038,0.963110,7915,155,554
2,0.685090,0.963085,7920,156,549
3,0.692986,0.963068,7911,154,558
4,0.691012,0.963047,7913,155,556
5,0.694960,0.962988,7910,154,559
6,0.698908,0.962894,7907,154,562
7,0.696934,0.962894,7907,154,562
8,0.683116,0.962878,7921,159,548
9,0.714699,0.962860,7894,145,575


In [22]:
# XGBoost Cell 7: Tune separate India and US thresholds for XGBoost so we can
# compare its best decoder fairly against the country-specific LightGBM result.

xgb_country_threshold_results = {}
xgb_country_best = {}

for country in ["India", "US"]:
    country_entity_mask = entity_country == country
    country_entity_indices = np.where(country_entity_mask)[0]

    global_to_local = {
        global_idx: local_idx
        for local_idx, global_idx in enumerate(country_entity_indices)
    }

    pair_mask = np.isin(entity_idx, country_entity_indices)

    country_scores = xgb_scores[pair_mask]
    country_labels = xgb_labels[pair_mask]
    country_global_idx = entity_idx[pair_mask]

    country_local_idx = np.array(
        [global_to_local[i] for i in country_global_idx],
        dtype=np.int32,
    )

    country_truth = total_truth[country_entity_indices]
    n_country_entities = len(country_entity_indices)

    rows = []

    for threshold in tqdm(
        thresholds,
        desc=f"XGBoost {country}",
        unit="threshold",
        dynamic_ncols=True,
    ):
        predicted = country_scores >= threshold

        pred_count = np.bincount(
            country_local_idx[predicted],
            minlength=n_country_entities,
        )

        tp_country = np.bincount(
            country_local_idx[predicted & (country_labels == 1)],
            minlength=n_country_entities,
        )

        fp_country = pred_count - tp_country
        fn_country = country_truth - tp_country

        denominator = (
            (1 + BETA2) * tp_country
            + BETA2 * fn_country
            + fp_country
        )

        entity_scores = np.zeros(
            n_country_entities,
            dtype=np.float64,
        )

        both_empty = (
            (country_truth == 0)
            & (pred_count == 0)
        )

        entity_scores[both_empty] = 1.0

        normal = denominator > 0

        entity_scores[normal] = (
            (1 + BETA2) * tp_country[normal]
            / denominator[normal]
        )

        rows.append({
            "threshold": float(threshold),
            "macro_f0.5": float(entity_scores.mean()),
            "tp": int(tp_country.sum()),
            "fp": int(fp_country.sum()),
            "fn": int(fn_country.sum()),
        })

    result = pd.DataFrame(rows)

    best = result.loc[
        result["macro_f0.5"].idxmax()
    ]

    xgb_country_threshold_results[country] = result
    xgb_country_best[country] = {
        "threshold": float(best["threshold"]),
        "macro_f0.5": float(best["macro_f0.5"]),
        "tp": int(best["tp"]),
        "fp": int(best["fp"]),
        "fn": int(best["fn"]),
    }

    print(f"\n🏆 XGBoost {country}")
    print(f"Best threshold: {best['threshold']:.6f}")
    print(f"Macro F0.5:     {best['macro_f0.5']:.6f}")
    print(f"TP:             {int(best['tp']):,}")
    print(f"FP:             {int(best['fp']):,}")
    print(f"FN:             {int(best['fn']):,}")

    display(
        result
        .sort_values("macro_f0.5", ascending=False)
        .head(5)
        .reset_index(drop=True)
    )

print("\nSelected XGBoost thresholds:")
print({
    country: values["threshold"]
    for country, values in xgb_country_best.items()
})

XGBoost India:   0%|          | 0/500 [00:00<?, ?threshold/s]


🏆 XGBoost India
Best threshold: 0.750230
Macro F0.5:     0.947698
TP:             2,978
FP:             64
FN:             326


,threshold,macro_f0.5,tp,fp,fn
0,0.750230,0.947698,2978,64,326
1,0.752204,0.947698,2978,64,326
2,0.758126,0.947584,2975,63,329
3,0.754178,0.947524,2977,64,327
4,0.762074,0.947520,2972,62,332


XGBoost US:   0%|          | 0/500 [00:00<?, ?threshold/s]


🏆 XGBoost US
Best threshold: 0.683116
Macro F0.5:     0.973457
TP:             4,922
FP:             79
FN:             243


,threshold,macro_f0.5,tp,fp,fn
0,0.683116,0.973457,4922,79,243
1,0.685090,0.973415,4921,79,244
2,0.712725,0.973412,4909,73,256
3,0.677194,0.973376,4923,81,242
4,0.714699,0.973370,4908,73,257



Selected XGBoost thresholds:
{'India': 0.7502304609218436, 'US': 0.6831162324649298}


In [23]:
# XGBoost Cell 8: Apply the best India/US XGBoost thresholds together and compute
# the combined validation macro-F0.5 for a fair comparison against LightGBM.

XGB_COUNTRY_THRESHOLDS = {
    "India": xgb_country_best["India"]["threshold"],
    "US": xgb_country_best["US"]["threshold"],
}

xgb_pair_thresholds = np.array(
    [XGB_COUNTRY_THRESHOLDS[c] for c in pair_countries],
    dtype=np.float32,
)

xgb_predicted = xgb_scores >= xgb_pair_thresholds

xgb_pred_count = np.bincount(
    entity_idx[xgb_predicted],
    minlength=n_entities,
)

xgb_tp_country_specific = np.bincount(
    entity_idx[xgb_predicted & (xgb_labels == 1)],
    minlength=n_entities,
)

xgb_fp_country_specific = xgb_pred_count - xgb_tp_country_specific
xgb_fn_country_specific = total_truth - xgb_tp_country_specific

xgb_denominator = (
    (1 + BETA2) * xgb_tp_country_specific
    + BETA2 * xgb_fn_country_specific
    + xgb_fp_country_specific
)

xgb_entity_f05_country_specific = np.zeros(
    n_entities,
    dtype=np.float64,
)

xgb_both_empty = (
    (total_truth == 0)
    & (xgb_pred_count == 0)
)

xgb_entity_f05_country_specific[xgb_both_empty] = 1.0

xgb_normal = xgb_denominator > 0

xgb_entity_f05_country_specific[xgb_normal] = (
    (1 + BETA2) * xgb_tp_country_specific[xgb_normal]
    / xgb_denominator[xgb_normal]
)

xgb_combined_score = xgb_entity_f05_country_specific.mean()

print("XGBoost country-specific thresholds:")
print(XGB_COUNTRY_THRESHOLDS)

print("\nCombined XGBoost validation results:")
print(f"Macro F0.5: {xgb_combined_score:.6f}")
print(f"TP:         {int(xgb_tp_country_specific.sum()):,}")
print(f"FP:         {int(xgb_fp_country_specific.sum()):,}")
print(f"FN:         {int(xgb_fn_country_specific.sum()):,}")

print("\nHead-to-head best decoder:")
print(f"LightGBM:   {combined_score:.6f}")
print(f"XGBoost:    {xgb_combined_score:.6f}")
print(f"Difference: {xgb_combined_score - combined_score:+.6f}")

XGBoost country-specific thresholds:
{'India': 0.7502304609218436, 'US': 0.6831162324649298}

Combined XGBoost validation results:
Macro F0.5: 0.963363
TP:         7,900
FP:         143
FN:         569

Head-to-head best decoder:
LightGBM:   0.963131
XGBoost:    0.963363
Difference: +0.000232


In [24]:
# XGBoost Cell 9: Compare LightGBM and XGBoost prediction disagreements at their
# own country-specific thresholds to measure ensemble complementarity.

# LightGBM decisions using its country-specific thresholds.
lgb_pair_thresholds = np.array(
    [COUNTRY_THRESHOLDS[c] for c in pair_countries],
    dtype=np.float32,
)

lgb_predicted = scores >= lgb_pair_thresholds

# XGBoost decisions using its own country-specific thresholds.
xgb_predicted = xgb_scores >= xgb_pair_thresholds

# Pair-level disagreement masks.
both_positive = lgb_predicted & xgb_predicted
lgb_only = lgb_predicted & ~xgb_predicted
xgb_only = xgb_predicted & ~lgb_predicted
both_negative = ~lgb_predicted & ~xgb_predicted

summary_rows = []

for name, mask in [
    ("both_positive", both_positive),
    ("lgb_only", lgb_only),
    ("xgb_only", xgb_only),
    ("both_negative", both_negative),
]:
    n_pairs = int(mask.sum())
    true_pairs = int(labels[mask].sum())
    false_pairs = n_pairs - true_pairs

    precision = (
        true_pairs / n_pairs
        if n_pairs > 0
        else np.nan
    )

    summary_rows.append({
        "group": name,
        "pairs": n_pairs,
        "true_matches": true_pairs,
        "false_matches": false_pairs,
        "precision": precision,
    })

disagreement_summary = pd.DataFrame(summary_rows)

display(disagreement_summary)

print("\nKey ensemble signal:")
print(
    "True matches found only by LightGBM:",
    int(labels[lgb_only].sum())
)
print(
    "False positives found only by LightGBM:",
    int((1 - labels[lgb_only]).sum())
)

print(
    "\nTrue matches found only by XGBoost:",
    int(labels[xgb_only].sum())
)
print(
    "False positives found only by XGBoost:",
    int((1 - labels[xgb_only]).sum())
)

print(
    "\nPrediction disagreements:",
    int((lgb_predicted != xgb_predicted).sum())
)

,group,pairs,true_matches,false_matches,precision
0,both_positive,7975,7853,122,0.984702
1,lgb_only,71,44,27,0.619718
2,xgb_only,68,47,21,0.691176
3,both_negative,110305,319,109986,0.002892



Key ensemble signal:
True matches found only by LightGBM: 44
False positives found only by LightGBM: 27

True matches found only by XGBoost: 47
False positives found only by XGBoost: 21

Prediction disagreements: 139


In [25]:
# Ensemble Cell 10: Sweep LightGBM/XGBoost probability blend weights and
# decision thresholds using the exact competition macro-F0.5 metric.
# No model is trained here; this tests whether their complementary errors combine.

blend_rows = []

BLEND_WEIGHTS = np.linspace(0.0, 1.0, 21)
# alpha = LightGBM weight
# 1-alpha = XGBoost weight

for alpha in tqdm(
    BLEND_WEIGHTS,
    desc="Testing LGBM/XGB blends",
    unit="blend",
    dynamic_ncols=True,
):
    blended_scores = (
        alpha * scores
        + (1.0 - alpha) * xgb_scores
    ).astype(np.float32)

    best_for_alpha = None

    for threshold in thresholds:
        predicted = blended_scores >= threshold

        pred_count = np.bincount(
            entity_idx[predicted],
            minlength=n_entities,
        )

        tp_blend = np.bincount(
            entity_idx[predicted & (labels == 1)],
            minlength=n_entities,
        )

        fp_blend = pred_count - tp_blend
        fn_blend = total_truth - tp_blend

        denominator = (
            (1 + BETA2) * tp_blend
            + BETA2 * fn_blend
            + fp_blend
        )

        entity_f05 = np.zeros(
            n_entities,
            dtype=np.float64,
        )

        both_empty = (
            (total_truth == 0)
            & (pred_count == 0)
        )

        entity_f05[both_empty] = 1.0

        normal = denominator > 0

        entity_f05[normal] = (
            (1 + BETA2) * tp_blend[normal]
            / denominator[normal]
        )

        macro_f05 = float(entity_f05.mean())

        if (
            best_for_alpha is None
            or macro_f05 > best_for_alpha["macro_f0.5"]
        ):
            best_for_alpha = {
                "lgb_weight": float(alpha),
                "xgb_weight": float(1.0 - alpha),
                "threshold": float(threshold),
                "macro_f0.5": macro_f05,
                "tp": int(tp_blend.sum()),
                "fp": int(fp_blend.sum()),
                "fn": int(fn_blend.sum()),
            }

    blend_rows.append(best_for_alpha)

blend_results = pd.DataFrame(blend_rows)

best_blend = blend_results.loc[
    blend_results["macro_f0.5"].idxmax()
]

print("\n🏆 BEST GLOBAL BLEND")
print(f"LightGBM weight: {best_blend['lgb_weight']:.2f}")
print(f"XGBoost weight:  {best_blend['xgb_weight']:.2f}")
print(f"Threshold:       {best_blend['threshold']:.6f}")
print(f"Macro F0.5:      {best_blend['macro_f0.5']:.6f}")
print(f"TP:              {int(best_blend['tp']):,}")
print(f"FP:              {int(best_blend['fp']):,}")
print(f"FN:              {int(best_blend['fn']):,}")

print("\nCurrent best standalone decoders:")
print(f"LightGBM country-specific: {combined_score:.6f}")
print(f"XGBoost country-specific:  {xgb_combined_score:.6f}")

print("\nTop blend weights:")
display(
    blend_results
    .sort_values("macro_f0.5", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

Testing LGBM/XGB blends:   0%|          | 0/21 [00:00<?, ?blend/s]


🏆 BEST GLOBAL BLEND
LightGBM weight: 0.50
XGBoost weight:  0.50
Threshold:       0.736413
Macro F0.5:      0.963573
TP:              7,871
FP:              129
FN:              598

Current best standalone decoders:
LightGBM country-specific: 0.963131
XGBoost country-specific:  0.963363

Top blend weights:


,lgb_weight,xgb_weight,threshold,macro_f0.5,tp,fp,fn
0,0.50,0.50,0.736413,0.963573,7871,129,598
1,0.55,0.45,0.738387,0.963469,7867,128,602
2,0.35,0.65,0.773918,0.963453,7843,117,626
3,0.45,0.55,0.728517,0.963444,7879,133,590
4,0.40,0.60,0.696934,0.963390,7904,143,565
5,0.65,0.35,0.687064,0.963384,7906,142,563
6,0.30,0.70,0.724569,0.963384,7885,137,584
7,0.25,0.75,0.742335,0.963348,7873,130,596
8,0.20,0.80,0.720621,0.963323,7889,140,580
9,0.70,0.30,0.696934,0.963322,7902,141,567


In [26]:
# Ensemble Cell 11: Tune LightGBM/XGBoost blend weights and thresholds separately
# for India and the US to exploit their different calibration behavior.

country_blend_results = {}
country_blend_best = {}

BLEND_WEIGHTS = np.linspace(0.0, 1.0, 21)

for country in ["India", "US"]:
    country_entity_mask = entity_country == country
    country_entity_indices = np.where(country_entity_mask)[0]

    global_to_local = {
        global_idx: local_idx
        for local_idx, global_idx in enumerate(country_entity_indices)
    }

    pair_mask = np.isin(entity_idx, country_entity_indices)

    country_lgb_scores = scores[pair_mask]
    country_xgb_scores = xgb_scores[pair_mask]
    country_labels = labels[pair_mask]
    country_global_idx = entity_idx[pair_mask]

    country_local_idx = np.array(
        [global_to_local[i] for i in country_global_idx],
        dtype=np.int32,
    )

    country_truth = total_truth[country_entity_indices]
    n_country_entities = len(country_entity_indices)

    rows = []

    for alpha in tqdm(
        BLEND_WEIGHTS,
        desc=f"Blend tuning {country}",
        unit="blend",
        dynamic_ncols=True,
    ):
        blended_scores = (
            alpha * country_lgb_scores
            + (1.0 - alpha) * country_xgb_scores
        ).astype(np.float32)

        best_for_alpha = None

        for threshold in thresholds:
            predicted = blended_scores >= threshold

            pred_count = np.bincount(
                country_local_idx[predicted],
                minlength=n_country_entities,
            )

            tp_blend = np.bincount(
                country_local_idx[predicted & (country_labels == 1)],
                minlength=n_country_entities,
            )

            fp_blend = pred_count - tp_blend
            fn_blend = country_truth - tp_blend

            denominator = (
                (1 + BETA2) * tp_blend
                + BETA2 * fn_blend
                + fp_blend
            )

            entity_f05 = np.zeros(
                n_country_entities,
                dtype=np.float64,
            )

            both_empty = (
                (country_truth == 0)
                & (pred_count == 0)
            )

            entity_f05[both_empty] = 1.0

            normal = denominator > 0

            entity_f05[normal] = (
                (1 + BETA2) * tp_blend[normal]
                / denominator[normal]
            )

            macro_f05 = float(entity_f05.mean())

            if (
                best_for_alpha is None
                or macro_f05 > best_for_alpha["macro_f0.5"]
            ):
                best_for_alpha = {
                    "country": country,
                    "lgb_weight": float(alpha),
                    "xgb_weight": float(1.0 - alpha),
                    "threshold": float(threshold),
                    "macro_f0.5": macro_f05,
                    "tp": int(tp_blend.sum()),
                    "fp": int(fp_blend.sum()),
                    "fn": int(fn_blend.sum()),
                }

        rows.append(best_for_alpha)

    result = pd.DataFrame(rows)

    best = result.loc[
        result["macro_f0.5"].idxmax()
    ]

    country_blend_results[country] = result
    country_blend_best[country] = best.to_dict()

    print(f"\n🏆 BEST BLEND — {country}")
    print(f"LightGBM weight: {best['lgb_weight']:.2f}")
    print(f"XGBoost weight:  {best['xgb_weight']:.2f}")
    print(f"Threshold:       {best['threshold']:.6f}")
    print(f"Macro F0.5:      {best['macro_f0.5']:.6f}")
    print(f"TP:              {int(best['tp']):,}")
    print(f"FP:              {int(best['fp']):,}")
    print(f"FN:              {int(best['fn']):,}")

    display(
        result
        .sort_values("macro_f0.5", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

Blend tuning India:   0%|          | 0/21 [00:00<?, ?blend/s]


🏆 BEST BLEND — India
LightGBM weight: 0.35
XGBoost weight:  0.65
Threshold:       0.773918
Macro F0.5:      0.948160
TP:              2,968
FP:              58
FN:              336


,country,lgb_weight,xgb_weight,threshold,macro_f0.5,tp,fp,fn
0,India,0.35,0.65,0.773918,0.948160,2968,58,336
1,India,0.50,0.50,0.734439,0.948086,2977,63,327
2,India,0.55,0.45,0.734439,0.948086,2977,63,327
3,India,0.10,0.90,0.738387,0.948048,2980,64,324
4,India,0.05,0.95,0.742335,0.948032,2980,64,324
5,India,0.20,0.80,0.760100,0.948019,2975,61,329
6,India,0.25,0.75,0.754178,0.947994,2974,61,330
7,India,0.15,0.85,0.754178,0.947925,2977,63,327
8,India,0.30,0.70,0.762074,0.947916,2971,60,333
9,India,0.45,0.55,0.728517,0.947894,2979,64,325


Blend tuning US:   0%|          | 0/21 [00:00<?, ?blend/s]


🏆 BEST BLEND — US
LightGBM weight: 0.90
XGBoost weight:  0.10
Threshold:       0.740361
Macro F0.5:      0.974230
TP:              4,891
FP:              64
FN:              274


,country,lgb_weight,xgb_weight,threshold,macro_f0.5,tp,fp,fn
0,US,0.90,0.10,0.740361,0.974230,4891,64,274
1,US,0.85,0.15,0.742335,0.974067,4890,65,275
2,US,0.65,0.35,0.687064,0.974065,4918,72,247
3,US,0.50,0.50,0.691012,0.973983,4919,73,246
4,US,0.45,0.55,0.691012,0.973950,4918,73,247
5,US,0.95,0.05,0.736413,0.973928,4892,67,273
6,US,0.70,0.30,0.696934,0.973921,4913,71,252
7,US,0.60,0.40,0.700882,0.973909,4913,71,252
8,US,0.40,0.60,0.696934,0.973880,4917,73,248
9,US,0.55,0.45,0.692986,0.973871,4918,73,247


In [28]:
# Ensemble Cell 12: Apply the best country-specific blend weights and thresholds
# simultaneously and compute the final combined validation macro-F0.5.

COUNTRY_BLEND_CONFIG = {
    "India": {
        "lgb_weight": float(country_blend_best["India"]["lgb_weight"]),
        "xgb_weight": float(country_blend_best["India"]["xgb_weight"]),
        "threshold": float(country_blend_best["India"]["threshold"]),
    },
    "US": {
        "lgb_weight": float(country_blend_best["US"]["lgb_weight"]),
        "xgb_weight": float(country_blend_best["US"]["xgb_weight"]),
        "threshold": float(country_blend_best["US"]["threshold"]),
    },
}

final_blend_scores = np.empty_like(scores, dtype=np.float32)
final_blend_thresholds = np.empty_like(scores, dtype=np.float32)

for country, cfg in COUNTRY_BLEND_CONFIG.items():
    mask = pair_countries == country

    final_blend_scores[mask] = (
        cfg["lgb_weight"] * scores[mask]
        + cfg["xgb_weight"] * xgb_scores[mask]
    )

    final_blend_thresholds[mask] = cfg["threshold"]

final_blend_predicted = (
    final_blend_scores >= final_blend_thresholds
)

final_pred_count = np.bincount(
    entity_idx[final_blend_predicted],
    minlength=n_entities,
)

final_tp = np.bincount(
    entity_idx[
        final_blend_predicted
        & (labels == 1)
    ],
    minlength=n_entities,
)

final_fp = final_pred_count - final_tp
final_fn = total_truth - final_tp

final_denominator = (
    (1 + BETA2) * final_tp
    + BETA2 * final_fn
    + final_fp
)

final_entity_f05 = np.zeros(
    n_entities,
    dtype=np.float64,
)

both_empty = (
    (total_truth == 0)
    & (final_pred_count == 0)
)

final_entity_f05[both_empty] = 1.0

normal = final_denominator > 0

final_entity_f05[normal] = (
    (1 + BETA2) * final_tp[normal]
    / final_denominator[normal]
)

final_blend_score = float(final_entity_f05.mean())

print("Country-specific blend config:")
print(COUNTRY_BLEND_CONFIG)

print("\n🏆 FINAL COUNTRY-SPECIFIC BLEND")
print(f"Macro F0.5: {final_blend_score:.6f}")
print(f"TP:         {int(final_tp.sum()):,}")
print(f"FP:         {int(final_fp.sum()):,}")
print(f"FN:         {int(final_fn.sum()):,}")

print("\nComparison:")
print(f"LightGBM country-specific: {combined_score:.6f}")
print(f"XGBoost country-specific:  {xgb_combined_score:.6f}")
print(f"Global 50/50 blend:        {best_blend['macro_f0.5']:.6f}")
print(f"Country-specific blend:    {final_blend_score:.6f}")

Country-specific blend config:
{'India': {'lgb_weight': 0.35000000000000003, 'xgb_weight': 0.6499999999999999, 'threshold': 0.7739178356713426}, 'US': {'lgb_weight': 0.9, 'xgb_weight': 0.09999999999999998, 'threshold': 0.7403607214428858}}

🏆 FINAL COUNTRY-SPECIFIC BLEND
Macro F0.5: 0.964014
TP:         7,859
FP:         122
FN:         610

Comparison:
LightGBM country-specific: 0.963131
XGBoost country-specific:  0.963363
Global 50/50 blend:        0.963573
Country-specific blend:    0.964014


## E2.5 — OOF Stacking Ensemble

In [29]:
# E2.5 Cell 1: Create 5 leakage-safe entity-level folds for OOF stacking.
# We stratify by country + ground-truth match-count bucket so every fold has
# a similar mix of India/US and zero/single/multi-match Source1 entities.

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5
OOF_SEED = 20260926

# Build one row per training Source1 entity.
train_entities_df = pd.DataFrame({
    "source1_entity_id": sorted(train_s1_ids)
})

# Country for each Source1 entity.
train_entities_df["country"] = (
    train_entities_df["source1_entity_id"]
    .map(country_map)
)

# Ground-truth match count for each Source1 entity.
train_gt_counts = (
    gt[gt["source1_entity_id"].isin(train_s1_ids)]
    .copy()
)

train_gt_counts["truth_count"] = (
    train_gt_counts["matched_entity_ids"]
    .fillna("")
    .astype(str)
    .apply(
        lambda x: 0
        if x.strip() == ""
        else len(x.split(","))
    )
)

truth_count_train_map = dict(
    zip(
        train_gt_counts["source1_entity_id"],
        train_gt_counts["truth_count"],
    )
)

train_entities_df["truth_count"] = (
    train_entities_df["source1_entity_id"]
    .map(truth_count_train_map)
    .fillna(0)
    .astype(int)
)

# Bucket match multiplicity to make stratification stable.
train_entities_df["match_bucket"] = pd.cut(
    train_entities_df["truth_count"],
    bins=[-1, 0, 1, 2, 4, 100],
    labels=["0", "1", "2", "3-4", "5+"],
)

train_entities_df["stratum"] = (
    train_entities_df["country"].astype(str)
    + "_"
    + train_entities_df["match_bucket"].astype(str)
)

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=OOF_SEED,
)

train_entities_df["fold"] = -1

for fold_id, (_, fold_val_idx) in enumerate(
    skf.split(
        train_entities_df,
        train_entities_df["stratum"],
    )
):
    train_entities_df.loc[
        fold_val_idx,
        "fold"
    ] = fold_id

assert (train_entities_df["fold"] >= 0).all()
assert train_entities_df["source1_entity_id"].nunique() == 7568

print("OOF entities:", len(train_entities_df))
print("Folds:", N_FOLDS)

print("\nEntities per fold:")
display(
    train_entities_df
    .groupby("fold")
    .size()
    .rename("entities")
    .reset_index()
)

print("\nCountry balance:")
display(
    pd.crosstab(
        train_entities_df["fold"],
        train_entities_df["country"],
    )
)

print("\nMatch-count balance:")
display(
    pd.crosstab(
        train_entities_df["fold"],
        train_entities_df["match_bucket"],
    )
)

print("✅ 5 leakage-safe OOF folds created")

OOF entities: 7568
Folds: 5

Entities per fold:


,fold,entities
0,0,1514
1,1,1514
2,2,1514
3,3,1513
4,4,1513



Country balance:


country,India,US
fold,,
0,602,912
1,602,912
2,603,911
3,601,912
4,601,912



Match-count balance:


match_bucket,0,1,2,3-4,5+
fold,,,,,
0,91,80,258,701,384
1,91,80,258,701,384
2,89,82,258,701,384
3,90,80,258,700,385
4,90,80,258,700,385


✅ 5 leakage-safe OOF folds created


In [30]:
# E2.5 Cell 2: Generate leakage-safe OOF probabilities from 5 LightGBM folds
# and 5 XGBoost folds. Each entity is scored only by models that did not train on it.

import lightgbm as lgb
import xgboost as xgb
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import time

# Training-candidate rows only; external validation remains untouched.
train_feature_rows = features[
    features["source1_entity_id"].isin(train_s1_ids)
].copy()

# Map each training entity to its OOF fold.
entity_fold_map = dict(
    zip(
        train_entities_df["source1_entity_id"],
        train_entities_df["fold"],
    )
)

train_feature_rows["oof_fold"] = (
    train_feature_rows["source1_entity_id"]
    .map(entity_fold_map)
    .astype("uint8")
)

# Preallocate OOF predictions.
oof_lgb = np.full(
    len(train_feature_rows),
    np.nan,
    dtype=np.float32,
)

oof_xgb = np.full(
    len(train_feature_rows),
    np.nan,
    dtype=np.float32,
)

# Keep the same core hyperparameters as the validated base models.
OOF_LGB_PARAMS = {
    "objective": "binary",
    "metric": ["binary_logloss", "auc"],
    "learning_rate": 0.05,
    "num_leaves": 63,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "min_data_in_leaf": 50,
    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed": 42,
    "verbosity": -1,
}

OOF_XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": ["logloss", "auc"],
    "learning_rate": 0.05,
    "max_depth": 8,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "max_bin": 256,
    "tree_method": "hist",
    "device": "cuda",
    "seed": 42,
}

fold_metrics = []

progress = tqdm(
    total=N_FOLDS * 2,
    desc="OOF base models",
    unit="model",
    dynamic_ncols=True,
)

t0 = time.time()

for fold_id in range(N_FOLDS):
    fold_train_mask = (
        train_feature_rows["oof_fold"] != fold_id
    )
    fold_val_mask = (
        train_feature_rows["oof_fold"] == fold_id
    )

    X_fold_train = train_feature_rows.loc[
        fold_train_mask,
        model_features,
    ]
    y_fold_train = train_feature_rows.loc[
        fold_train_mask,
        "label",
    ]

    X_fold_val = train_feature_rows.loc[
        fold_val_mask,
        model_features,
    ]
    y_fold_val = train_feature_rows.loc[
        fold_val_mask,
        "label",
    ]

    # ----- LightGBM -----
    lgb_train = lgb.Dataset(
        X_fold_train,
        label=y_fold_train,
        feature_name=model_features,
    )

    lgb_valid = lgb.Dataset(
        X_fold_val,
        label=y_fold_val,
        reference=lgb_train,
        feature_name=model_features,
    )

    fold_lgb_model = lgb.train(
        OOF_LGB_PARAMS,
        lgb_train,
        num_boost_round=2000,
        valid_sets=[lgb_valid],
        valid_names=["validation"],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=False,
            ),
            lgb.log_evaluation(period=0),
        ],
    )

    lgb_fold_pred = fold_lgb_model.predict(
        X_fold_val,
        num_iteration=fold_lgb_model.best_iteration,
    )

    oof_lgb[fold_val_mask.to_numpy()] = (
        lgb_fold_pred.astype(np.float32)
    )

    progress.update(1)
    progress.set_postfix(
        fold=fold_id,
        model="LightGBM",
        best_iter=fold_lgb_model.best_iteration,
    )

    # ----- XGBoost -----
    dtrain_fold = xgb.QuantileDMatrix(
        X_fold_train,
        y_fold_train,
        max_bin=256,
        feature_names=model_features,
    )

    dval_fold = xgb.QuantileDMatrix(
        X_fold_val,
        y_fold_val,
        max_bin=256,
        feature_names=model_features,
        ref=dtrain_fold,
    )

    fold_xgb_model = xgb.train(
        params=OOF_XGB_PARAMS,
        dtrain=dtrain_fold,
        num_boost_round=2000,
        evals=[(dval_fold, "validation")],
        early_stopping_rounds=100,
        verbose_eval=False,
    )

    xgb_fold_pred = fold_xgb_model.predict(
        dval_fold,
        iteration_range=(
            0,
            fold_xgb_model.best_iteration + 1,
        ),
    )

    oof_xgb[fold_val_mask.to_numpy()] = (
        xgb_fold_pred.astype(np.float32)
    )

    progress.update(1)
    progress.set_postfix(
        fold=fold_id,
        model="XGBoost",
        best_iter=fold_xgb_model.best_iteration,
    )

    fold_metrics.append({
        "fold": fold_id,
        "validation_pairs": int(fold_val_mask.sum()),
        "validation_positives": int(y_fold_val.sum()),
        "lgb_best_iteration": int(
            fold_lgb_model.best_iteration
        ),
        "xgb_best_iteration": int(
            fold_xgb_model.best_iteration
        ),
    })

progress.close()

elapsed = time.time() - t0

assert not np.isnan(oof_lgb).any()
assert not np.isnan(oof_xgb).any()

train_feature_rows["oof_lgb_score"] = oof_lgb
train_feature_rows["oof_xgb_score"] = oof_xgb

print(f"\nOOF generation time: {elapsed:.1f}s")
print("OOF rows:", len(train_feature_rows))
print(
    "LightGBM score range:",
    float(oof_lgb.min()),
    "→",
    float(oof_lgb.max()),
)
print(
    "XGBoost score range:",
    float(oof_xgb.min()),
    "→",
    float(oof_xgb.max()),
)

display(pd.DataFrame(fold_metrics))

print("\n✅ Leakage-safe OOF base predictions generated")

OOF base models:   0%|          | 0/10 [00:00<?, ?model/s]


OOF generation time: 37.5s
OOF rows: 369671
LightGBM score range: 9.255592772206e-07 → 0.9999799728393555
XGBoost score range: 9.294379310631484e-08 → 0.9999940395355225


,fold,validation_pairs,validation_positives,lgb_best_iteration,xgb_best_iteration
0,0,74120,5088,296,286
1,1,73977,5066,225,218
2,2,73948,5081,284,614
3,3,73907,5089,244,359
4,4,73719,5080,300,309



✅ Leakage-safe OOF base predictions generated


In [31]:
# E2.5 Cell 3: Train a leakage-safe Logistic Regression stacker on OOF
# LightGBM/XGBoost predictions, then score the untouched external validation set.

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

# Helper: convert probabilities to stable logits.
def safe_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

# ----- Build OOF meta-training features -----

oof_country = (
    train_feature_rows["source1_entity_id"]
    .map(country_map)
    .to_numpy()
)

oof_meta = pd.DataFrame({
    "lgb_score": oof_lgb,
    "xgb_score": oof_xgb,

    # Logit versions help Logistic Regression combine very confident scores.
    "lgb_logit": safe_logit(oof_lgb),
    "xgb_logit": safe_logit(oof_xgb),

    # Explicit disagreement / agreement signals.
    "score_mean": (oof_lgb + oof_xgb) / 2.0,
    "score_abs_diff": np.abs(oof_lgb - oof_xgb),
    "score_product": oof_lgb * oof_xgb,

    # Open-set-safe training-country indicator.
    # France will later map to 0 for both India/US flags unless we choose otherwise.
    "is_india": (oof_country == "India").astype(np.uint8),
    "is_us": (oof_country == "US").astype(np.uint8),
})

oof_target = train_feature_rows["label"].to_numpy(dtype=np.uint8)

# ----- Train stacker -----

stacker = LogisticRegression(
    C=1.0,
    class_weight=None,
    max_iter=2000,
    solver="lbfgs",
    random_state=42,
)

print("🔥 TRAINING LOGISTIC REGRESSION STACKER")
print("OOF training pairs:", len(oof_meta))
print("Meta features:", oof_meta.shape[1])

stacker.fit(oof_meta, oof_target)

# ----- Prepare untouched validation meta-features -----

val_country = pair_countries

val_meta = pd.DataFrame({
    "lgb_score": scores,
    "xgb_score": xgb_scores,
    "lgb_logit": safe_logit(scores),
    "xgb_logit": safe_logit(xgb_scores),
    "score_mean": (scores + xgb_scores) / 2.0,
    "score_abs_diff": np.abs(scores - xgb_scores),
    "score_product": scores * xgb_scores,
    "is_india": (val_country == "India").astype(np.uint8),
    "is_us": (val_country == "US").astype(np.uint8),
})

stack_val_scores = stacker.predict_proba(val_meta)[:, 1].astype(np.float32)

print("\n✅ STACKER TRAINED")

print(
    "Validation ROC-AUC:",
    f"{roc_auc_score(labels, stack_val_scores):.6f}",
)

print(
    "Validation logloss:",
    f"{log_loss(labels, stack_val_scores):.6f}",
)

print(
    "Score range:",
    f"{stack_val_scores.min():.8f}",
    "→",
    f"{stack_val_scores.max():.8f}",
)

coef_table = pd.DataFrame({
    "feature": oof_meta.columns,
    "coefficient": stacker.coef_[0],
}).sort_values(
    "coefficient",
    key=np.abs,
    ascending=False,
)

print("\nStacker coefficients:")
display(coef_table.reset_index(drop=True))

🔥 TRAINING LOGISTIC REGRESSION STACKER
OOF training pairs: 369671
Meta features: 9

✅ STACKER TRAINED
Validation ROC-AUC: 0.999552
Validation logloss: 0.011535
Score range: 0.00000357 → 0.99995315

Stacker coefficients:


,feature,coefficient
0,lgb_logit,0.474593
1,xgb_logit,0.467467
2,is_us,0.036268
3,is_india,0.009607
4,score_abs_diff,0.004592
5,xgb_score,0.004134
6,score_mean,0.004113
7,lgb_score,0.004093
8,score_product,-0.002719


In [32]:
# E2.5 Cell 4: Sweep Logistic Regression stacker thresholds using the exact
# competition macro-F0.5 metric and compare against the current best ensemble.

stack_threshold_results = []

for threshold in tqdm(
    thresholds,
    desc="Sweeping stacker F0.5 thresholds",
    unit="threshold",
    dynamic_ncols=True,
):
    predicted = stack_val_scores >= threshold

    pred_count = np.bincount(
        entity_idx[predicted],
        minlength=n_entities,
    )

    tp_stack = np.bincount(
        entity_idx[predicted & (labels == 1)],
        minlength=n_entities,
    )

    fp_stack = pred_count - tp_stack
    fn_stack = total_truth - tp_stack

    denominator = (
        (1 + BETA2) * tp_stack
        + BETA2 * fn_stack
        + fp_stack
    )

    entity_f05_stack = np.zeros(
        n_entities,
        dtype=np.float64,
    )

    both_empty = (
        (total_truth == 0)
        & (pred_count == 0)
    )

    entity_f05_stack[both_empty] = 1.0

    normal = denominator > 0

    entity_f05_stack[normal] = (
        (1 + BETA2) * tp_stack[normal]
        / denominator[normal]
    )

    stack_threshold_results.append({
        "threshold": float(threshold),
        "macro_f0.5": float(entity_f05_stack.mean()),
        "tp": int(tp_stack.sum()),
        "fp": int(fp_stack.sum()),
        "fn": int(fn_stack.sum()),
    })

stack_threshold_results = pd.DataFrame(
    stack_threshold_results
)

stack_best_row = stack_threshold_results.loc[
    stack_threshold_results["macro_f0.5"].idxmax()
]

print("\n🏆 BEST LOGISTIC STACKER")
print(f"Threshold:  {stack_best_row['threshold']:.6f}")
print(f"Macro F0.5: {stack_best_row['macro_f0.5']:.6f}")
print(f"TP:         {int(stack_best_row['tp']):,}")
print(f"FP:         {int(stack_best_row['fp']):,}")
print(f"FN:         {int(stack_best_row['fn']):,}")

print("\nComparison:")
print(f"LightGBM country-specific: {combined_score:.6f}")
print(f"XGBoost country-specific:  {xgb_combined_score:.6f}")
print(f"Global 50/50 blend:        {best_blend['macro_f0.5']:.6f}")
print(f"Country-specific blend:    {final_blend_score:.6f}")
print(f"OOF Logistic stacker:      {stack_best_row['macro_f0.5']:.6f}")

print("\nTop stacker thresholds:")
display(
    stack_threshold_results
    .sort_values("macro_f0.5", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

Sweeping stacker F0.5 thresholds:   0%|          | 0/500 [00:00<?, ?threshold/s]


🏆 BEST LOGISTIC STACKER
Threshold:  0.744309
Macro F0.5: 0.963375
TP:         7,870
FP:         130
FN:         599

Comparison:
LightGBM country-specific: 0.963131
XGBoost country-specific:  0.963363
Global 50/50 blend:        0.963573
Country-specific blend:    0.964014
OOF Logistic stacker:      0.963375

Top stacker thresholds:


,threshold,macro_f0.5,tp,fp,fn
0,0.744309,0.963375,7870,130,599
1,0.742335,0.963364,7872,131,597
2,0.734439,0.963356,7880,135,589
3,0.746283,0.963338,7869,130,600
4,0.740361,0.963330,7874,132,595
5,0.736413,0.963328,7876,133,593
6,0.732465,0.963302,7882,137,587
7,0.738387,0.963290,7875,133,594
8,0.748257,0.963275,7867,130,602
9,0.766022,0.963236,7850,123,619


In [33]:
# E2.5 Cell 5: Tune separate India and US thresholds for the leakage-safe
# Logistic Regression stacker and compare against the 0.964014 blend benchmark.

stack_country_results = {}
stack_country_best = {}

for country in ["India", "US"]:
    country_entity_mask = entity_country == country
    country_entity_indices = np.where(country_entity_mask)[0]

    global_to_local = {
        global_idx: local_idx
        for local_idx, global_idx in enumerate(country_entity_indices)
    }

    pair_mask = np.isin(entity_idx, country_entity_indices)

    country_scores = stack_val_scores[pair_mask]
    country_labels = labels[pair_mask]
    country_global_idx = entity_idx[pair_mask]

    country_local_idx = np.array(
        [global_to_local[i] for i in country_global_idx],
        dtype=np.int32,
    )

    country_truth = total_truth[country_entity_indices]
    n_country_entities = len(country_entity_indices)

    rows = []

    for threshold in tqdm(
        thresholds,
        desc=f"Stacker {country}",
        unit="threshold",
        dynamic_ncols=True,
    ):
        predicted = country_scores >= threshold

        pred_count = np.bincount(
            country_local_idx[predicted],
            minlength=n_country_entities,
        )

        tp_country = np.bincount(
            country_local_idx[
                predicted & (country_labels == 1)
            ],
            minlength=n_country_entities,
        )

        fp_country = pred_count - tp_country
        fn_country = country_truth - tp_country

        denominator = (
            (1 + BETA2) * tp_country
            + BETA2 * fn_country
            + fp_country
        )

        entity_f05 = np.zeros(
            n_country_entities,
            dtype=np.float64,
        )

        both_empty = (
            (country_truth == 0)
            & (pred_count == 0)
        )
        entity_f05[both_empty] = 1.0

        normal = denominator > 0

        entity_f05[normal] = (
            (1 + BETA2) * tp_country[normal]
            / denominator[normal]
        )

        rows.append({
            "threshold": float(threshold),
            "macro_f0.5": float(entity_f05.mean()),
            "tp": int(tp_country.sum()),
            "fp": int(fp_country.sum()),
            "fn": int(fn_country.sum()),
        })

    result = pd.DataFrame(rows)

    best = result.loc[
        result["macro_f0.5"].idxmax()
    ]

    stack_country_results[country] = result
    stack_country_best[country] = best.to_dict()

    print(f"\n🏆 STACKER — {country}")
    print(f"Threshold:  {best['threshold']:.6f}")
    print(f"Macro F0.5: {best['macro_f0.5']:.6f}")
    print(f"TP:         {int(best['tp']):,}")
    print(f"FP:         {int(best['fp']):,}")
    print(f"FN:         {int(best['fn']):,}")

    display(
        result
        .sort_values("macro_f0.5", ascending=False)
        .head(5)
        .reset_index(drop=True)
    )

Stacker India:   0%|          | 0/500 [00:00<?, ?threshold/s]


🏆 STACKER — India
Threshold:  0.736413
Macro F0.5: 0.947786
TP:         2,977
FP:         64
FN:         327


,threshold,macro_f0.5,tp,fp,fn
0,0.736413,0.947786,2977,64,327
1,0.742335,0.947691,2976,64,328
2,0.740361,0.947691,2976,64,328
3,0.738387,0.947691,2976,64,328
4,0.744309,0.947691,2976,64,328


Stacker US:   0%|          | 0/500 [00:00<?, ?threshold/s]


🏆 STACKER — US
Threshold:  0.700882
Macro F0.5: 0.973983
TP:         4,919
FP:         73
FN:         246


,threshold,macro_f0.5,tp,fp,fn
0,0.700882,0.973983,4919,73,246
1,0.702856,0.973950,4918,73,247
2,0.706804,0.973950,4918,73,247
3,0.704830,0.973950,4918,73,247
4,0.710752,0.973915,4917,73,248


In [34]:
# E2.5 Cell 6: Apply the best India/US stacker thresholds together and compute
# the exact combined macro-F0.5 before deciding whether to keep or reject stacking.

STACK_COUNTRY_THRESHOLDS = {
    "India": float(stack_country_best["India"]["threshold"]),
    "US": float(stack_country_best["US"]["threshold"]),
}

stack_pair_thresholds = np.array(
    [STACK_COUNTRY_THRESHOLDS[c] for c in pair_countries],
    dtype=np.float32,
)

stack_country_predicted = (
    stack_val_scores >= stack_pair_thresholds
)

stack_pred_count = np.bincount(
    entity_idx[stack_country_predicted],
    minlength=n_entities,
)

stack_tp_country_specific = np.bincount(
    entity_idx[
        stack_country_predicted
        & (labels == 1)
    ],
    minlength=n_entities,
)

stack_fp_country_specific = (
    stack_pred_count - stack_tp_country_specific
)

stack_fn_country_specific = (
    total_truth - stack_tp_country_specific
)

stack_denominator = (
    (1 + BETA2) * stack_tp_country_specific
    + BETA2 * stack_fn_country_specific
    + stack_fp_country_specific
)

stack_entity_f05_country_specific = np.zeros(
    n_entities,
    dtype=np.float64,
)

both_empty = (
    (total_truth == 0)
    & (stack_pred_count == 0)
)

stack_entity_f05_country_specific[both_empty] = 1.0

normal = stack_denominator > 0

stack_entity_f05_country_specific[normal] = (
    (1 + BETA2)
    * stack_tp_country_specific[normal]
    / stack_denominator[normal]
)

stack_country_score = float(
    stack_entity_f05_country_specific.mean()
)

print("Stacker country thresholds:")
print(STACK_COUNTRY_THRESHOLDS)

print("\nCombined stacker results:")
print(f"Macro F0.5: {stack_country_score:.6f}")
print(f"TP:         {int(stack_tp_country_specific.sum()):,}")
print(f"FP:         {int(stack_fp_country_specific.sum()):,}")
print(f"FN:         {int(stack_fn_country_specific.sum()):,}")

print("\nCurrent leaderboard:")
print(f"LightGBM country-specific: {combined_score:.6f}")
print(f"XGBoost country-specific:  {xgb_combined_score:.6f}")
print(f"Global 50/50 blend:        {best_blend['macro_f0.5']:.6f}")
print(f"Country-specific blend:    {final_blend_score:.6f}")
print(f"OOF stacker global:        {stack_best_row['macro_f0.5']:.6f}")
print(f"OOF stacker country:       {stack_country_score:.6f}")

Stacker country thresholds:
{'India': 0.7364128256513026, 'US': 0.7008817635270541}

Combined stacker results:
Macro F0.5: 0.963718
TP:         7,896
FP:         137
FN:         573

Current leaderboard:
LightGBM country-specific: 0.963131
XGBoost country-specific:  0.963363
Global 50/50 blend:        0.963573
Country-specific blend:    0.964014
OOF stacker global:        0.963375
OOF stacker country:       0.963718


## E2.6 — Multilingual Embedding Retrieval

In [39]:
# E2.6 Cell 1: Check whether the required embedding/ANN libraries are installed
# before we build multilingual semantic retrieval on the V100.

packages = {}

for name in [
    "sentence_transformers",
    "transformers",
    "faiss",
]:
    try:
        module = __import__(name)
        packages[name] = getattr(module, "__version__", "installed")
    except Exception as e:
        packages[name] = f"NOT AVAILABLE: {type(e).__name__}: {e}"

for name, status in packages.items():
    print(f"{name:24s} {status}")

sentence_transformers    5.1.2
transformers             4.57.6
faiss                    NOT AVAILABLE: ModuleNotFoundError: No module named 'faiss'


In [40]:
# E2.6 Cell 2: Load multilingual-e5-base on the V100 and verify the model
# can encode a small batch before we launch the expensive semantic retrieval stage.

import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "intfloat/multilingual-e5-base"

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device="cuda",
)

test_texts = [
    "query: lakshmi traders 12 mg road bengaluru india",
    "passage: laxmi traders no 12 m g rd bangalore india",
    "passage: paris bakery rue de lyon france",
]

test_embeddings = embed_model.encode(
    test_texts,
    batch_size=3,
    normalize_embeddings=True,
    convert_to_tensor=True,
    show_progress_bar=True,
)

print("Embedding shape:", tuple(test_embeddings.shape))
print("Device:", test_embeddings.device)

similarities = (
    test_embeddings[0:1]
    @ test_embeddings[1:].T
)

print("Example similarities:")
print(similarities.detach().cpu().numpy())

print("✅ Multilingual E5 loaded and GPU encoding works")

CUDA available: True
GPU: Tesla V100-PCIE-32GB


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

... (3 outputs left out)

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (3, 768)
Device: cuda:0
Example similarities:
[[0.8752432 0.7898984]]
✅ Multilingual E5 loaded and GPU encoding works


In [42]:
# E2.6 Cell 3A: Reload the normalized E2.3 Source1 and feed tables after the
# kernel restart so we can inspect the exact columns used for embedding text.

import pandas as pd

SOURCE1_PATH = (
    REPO
    / "cache"
    / "normalized"
    / "e21_lgbm_c449c0c6b281_source1.parquet"
)

FEED_PATH = (
    REPO
    / "cache"
    / "normalized"
    / "e21_lgbm_c449c0c6b281_feed.parquet"
)

source1 = pd.read_parquet(SOURCE1_PATH)
feed = pd.read_parquet(FEED_PATH)

print("Source1:", source1.shape)
print("Feed:", feed.shape)

print("\nSource1 columns:")
print(source1.columns.tolist())

print("\nFeed columns:")
print(feed.columns.tolist())

print("\nSource1 sample:")
display(source1.head(3))

print("\nFeed sample:")
display(feed.head(3))

Source1: (10000, 18)
Feed: (1034502, 23)

Source1 columns:
['entity_id', 'business_name', 'business_address', 'country', 'name_raw', 'name_full', 'name_core', 'name_sorted', 'legal_form', 'name_acronym', 'name_tokens', 'address_raw', 'address_full', 'address_core', 'postcode', 'address_tokens', 'address_numeric_tokens', 'country_norm']

Feed columns:
['entity_id', 'business_name', 'business_address', 'country', '_sample_hash', '_sample_hash2', '_is_distractor', '_distractor_rank', '_feed_source', 'name_raw', 'name_full', 'name_core', 'name_sorted', 'legal_form', 'name_acronym', 'name_tokens', 'address_raw', 'address_full', 'address_core', 'postcode', 'address_tokens', 'address_numeric_tokens', 'country_norm']

Source1 sample:


,entity_id,business_name,business_address,country,name_raw,name_full,name_core,name_sorted,legal_form,name_acronym,name_tokens,address_raw,address_full,address_core,postcode,address_tokens,address_numeric_tokens,country_norm
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,Orelee's Barbershop,orelee s barbershop,orelee s barbershop,barbershop orelee s,,osb,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,1795 westchester drive high point nc,,1795 westchester drive high point nc,1795,us
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,Prime Money,prime money,prime money,money prime,,pm,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,17560 ellis road tahlequah ok,17560,17560 ellis road tahlequah ok,17560,us
2,S1-865131206,Helios,"66 Edgewood Street, Bridgeport, CT",US,Helios,helios,helios,helios,,,helios,"66 Edgewood Street, Bridgeport, CT",66 edgewood street bridgeport ct,66 edgewood street bridgeport ct,,66 edgewood street bridgeport ct,66,us



Feed sample:


,entity_id,business_name,business_address,country,_sample_hash,_sample_hash2,_is_distractor,_distractor_rank,_feed_source,name_raw,...,legal_form,name_acronym,name_tokens,address_raw,address_full,address_core,postcode,address_tokens,address_numeric_tokens,country_norm
0,S2-895116270,BELL BUILDERS CENTER LTD,"LIG 3, B2 -004, SURYA NAGARA, CHANDAPURA ANEKA...",India,0,0,0,0,source2,BELL BUILDERS CENTER LTD,...,ltd,bbc,bell builders center ltd,"LIG 3, B2 -004, SURYA NAGARA, CHANDAPURA ANEKA...",lig 3 b2 004 surya nagara chandapura anekal tk...,lig 3 b2 004 surya nagara chandapura anekal tk...,,lig 3 b2 004 surya nagara chandapura anekal tk...,3 004,india
1,S2-528205184,Visage Nursing Limited Home,"##464 , LANE NO. 11, GANDHI COLONY, MUZAFFARNA...",India,0,0,0,0,source2,Visage Nursing Limited Home,...,,vnlh,visage nursing limited home,"##464 , LANE NO. 11, GANDHI COLONY, MUZAFFARNA...",464 lane no 11 gandhi colony muzaffarnagar utt...,464 lane no 11 gandhi colony muzaffarnagar utt...,,464 lane no 11 gandhi colony muzaffarnagar utt...,464 11,india
2,S2-483453128,ryder development,"27642C MESABE DRIVE, MAGNOIA TOWNSHIP, TX",US,0,0,0,0,source2,ryder development,...,,rd,ryder development,"27642C MESABE DRIVE, MAGNOIA TOWNSHIP, TX",27642c mesabe drive magnoia township tx,27642c mesabe drive magnoia township tx,27642,27642c mesabe drive magnoia township tx,27642c,us


In [43]:
# E2.6 Cell 4: Build the structured multilingual-E5 text representation for
# Source1 queries and feed passages using normalized name, address, and country.

def build_e5_text(df, prefix):
    name = df["name_full"].fillna("").astype(str).str.strip()
    address = df["address_full"].fillna("").astype(str).str.strip()
    country = df["country_norm"].fillna("").astype(str).str.strip()

    return (
        prefix
        + "business name: " + name
        + " | address: " + address
        + " | country: " + country
    )

source1_e5_text = build_e5_text(
    source1,
    "query: ",
)

feed_e5_text = build_e5_text(
    feed,
    "passage: ",
)

print("Source1 texts:", len(source1_e5_text))
print("Feed texts:", len(feed_e5_text))

print("\nExample Source1 text:")
print(source1_e5_text.iloc[0])

print("\nExample feed text:")
print(feed_e5_text.iloc[0])

print("\nCharacter-length statistics:")
print(
    pd.DataFrame({
        "source1": source1_e5_text.str.len().describe(),
        "feed": feed_e5_text.str.len().describe(),
    })
)

print("\n✅ E5 structured texts prepared")

Source1 texts: 10000
Feed texts: 1034502

Example Source1 text:
query: business name: orelee s barbershop | address: 1795 westchester drive high point nc | country: us

Example feed text:
passage: business name: bell builders center ltd | address: lig 3 b2 004 surya nagara chandapura anekal tk bangalore karnataka | country: india

Character-length statistics:
            source1          feed
count  10000.000000  1.034502e+06
mean     121.645200  1.186706e+02
std       27.607719  2.507389e+01
min       73.000000  5.100000e+01
25%      100.000000  1.020000e+02
50%      112.000000  1.130000e+02
75%      141.000000  1.320000e+02
max      250.000000  3.030000e+02

✅ E5 structured texts prepared


In [44]:
# E2.6 Cell 5: Check token-length distributions for the structured E5 texts
# so we can choose a safe max sequence length before encoding ~1.04M records.

from tqdm.auto import tqdm
import numpy as np

tokenizer = embed_model.tokenizer

# Sample a large subset of the feed rather than tokenizing all 1.03M rows just
# for diagnostics. Include all Source1 texts because there are only 10k.
FEED_TOKEN_SAMPLE = 100_000

rng = np.random.default_rng(42)

sample_idx = rng.choice(
    len(feed_e5_text),
    size=min(FEED_TOKEN_SAMPLE, len(feed_e5_text)),
    replace=False,
)

feed_text_sample = feed_e5_text.iloc[sample_idx].tolist()

def get_token_lengths(texts, batch_size=2048):
    lengths = []

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Checking token lengths",
        unit="batch",
        dynamic_ncols=True,
    ):
        batch = texts[start:start + batch_size]

        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            truncation=False,
            padding=False,
            return_length=True,
        )

        lengths.extend(encoded["length"])

    return np.asarray(lengths, dtype=np.int32)

source1_token_lengths = get_token_lengths(
    source1_e5_text.tolist()
)

feed_token_lengths = get_token_lengths(
    feed_text_sample
)

def summarize_lengths(name, arr):
    print(f"\n{name}")
    print(f"count: {len(arr):,}")
    print(f"mean:  {arr.mean():.1f}")
    print(f"p50:   {np.percentile(arr, 50):.0f}")
    print(f"p95:   {np.percentile(arr, 95):.0f}")
    print(f"p99:   {np.percentile(arr, 99):.0f}")
    print(f"max:   {arr.max():,}")

summarize_lengths(
    "Source1 token lengths",
    source1_token_lengths,
)

summarize_lengths(
    "Feed token lengths (100k sample)",
    feed_token_lengths,
)

print("\nModel max sequence length:", embed_model.max_seq_length)

Checking token lengths:   0%|          | 0/5 [00:00<?, ?batch/s]

Checking token lengths:   0%|          | 0/49 [00:00<?, ?batch/s]


Source1 token lengths
count: 10,000
mean:  37.3
p50:   35
p95:   51
p99:   57
max:   77

Feed token lengths (100k sample)
count: 100,000
mean:  36.4
p50:   35
p95:   52
p99:   60
max:   83

Model max sequence length: 512


In [45]:
# E2.6 Cell 6: Encode all Source1 and feed texts with multilingual-E5 on the V100
# and checkpoint normalized float16 embeddings incrementally to disk with tqdm.

import numpy as np
import torch
from pathlib import Path
from tqdm.auto import tqdm
import time

embed_model.max_seq_length = 128

EMBED_DIR = REPO / "cache" / "embeddings"
EMBED_DIR.mkdir(parents=True, exist_ok=True)

S1_EMBED_PATH = (
    EMBED_DIR / "e26_multilingual_e5_source1_f16.npy"
)

FEED_EMBED_PATH = (
    EMBED_DIR / "e26_multilingual_e5_feed_f16.npy"
)

EMBED_DIM = 768
BATCH_SIZE = 512


def encode_to_memmap(text_series, output_path, desc):
    n = len(text_series)

    mmap = np.lib.format.open_memmap(
        output_path,
        mode="w+",
        dtype=np.float16,
        shape=(n, EMBED_DIM),
    )

    t0 = time.time()

    for start in tqdm(
        range(0, n, BATCH_SIZE),
        desc=desc,
        unit="batch",
        dynamic_ncols=True,
    ):
        end = min(start + BATCH_SIZE, n)

        batch_texts = text_series.iloc[start:end].tolist()

        with torch.inference_mode():
            batch_embeddings = embed_model.encode(
                batch_texts,
                batch_size=BATCH_SIZE,
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False,
            )

        mmap[start:end] = batch_embeddings.astype(
            np.float16,
            copy=False,
        )

        # Periodically flush so progress survives an interruption.
        if (start // BATCH_SIZE) % 100 == 0:
            mmap.flush()

    mmap.flush()

    elapsed = time.time() - t0

    print(
        f"{desc} complete: {n:,} rows "
        f"in {elapsed / 60:.2f} min"
    )
    print(
        f"Saved: {output_path} "
        f"({output_path.stat().st_size / 1024**3:.2f} GiB)"
    )

    return mmap


print("Embedding model:", EMBED_MODEL_NAME)
print("Max sequence length:", embed_model.max_seq_length)
print("Batch size:", BATCH_SIZE)
print("GPU:", torch.cuda.get_device_name(0))
print()

source1_embeddings = encode_to_memmap(
    source1_e5_text,
    S1_EMBED_PATH,
    "Encoding Source1",
)

feed_embeddings = encode_to_memmap(
    feed_e5_text,
    FEED_EMBED_PATH,
    "Encoding feed",
)

print("\nShapes:")
print("Source1:", source1_embeddings.shape)
print("Feed:   ", feed_embeddings.shape)

print("\n✅ E2.6 multilingual embeddings encoded and checkpointed")

Embedding model: intfloat/multilingual-e5-base
Max sequence length: 128
Batch size: 512
GPU: Tesla V100-PCIE-32GB



Encoding Source1:   0%|          | 0/20 [00:00<?, ?batch/s]

Encoding Source1 complete: 10,000 rows in 0.17 min
Saved: /home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/embeddings/e26_multilingual_e5_source1_f16.npy (0.01 GiB)


Encoding feed:   0%|          | 0/2021 [00:00<?, ?batch/s]

Encoding feed complete: 1,034,502 rows in 18.76 min
Saved: /home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/embeddings/e26_multilingual_e5_feed_f16.npy (1.48 GiB)

Shapes:
Source1: (10000, 768)
Feed:    (1034502, 768)

✅ E2.6 multilingual embeddings encoded and checkpointed


In [46]:
# E2.6 Cell 7: Run exact country-restricted multilingual-E5 top-20 retrieval
# on the V100 in chunks, keeping only the best semantic matches per Source1 entity.

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import time

E5_TOP_K = 20
QUERY_BATCH = 512
FEED_CHUNK = 100_000

device = torch.device("cuda")

source1_country = source1["country_norm"].fillna("").to_numpy()
feed_country = feed["country_norm"].fillna("").to_numpy()

all_result_rows = []

t0 = time.time()

for country in sorted(np.unique(source1_country)):
    q_idx = np.flatnonzero(source1_country == country)
    f_idx = np.flatnonzero(feed_country == country)

    print(
        f"\n🌍 {country.upper()}: "
        f"{len(q_idx):,} Source1 × {len(f_idx):,} feed records"
    )

    # Keep the running global top-K for every query in this country.
    best_scores = torch.full(
        (len(q_idx), E5_TOP_K),
        -float("inf"),
        dtype=torch.float32,
        device=device,
    )

    best_indices = torch.full(
        (len(q_idx), E5_TOP_K),
        -1,
        dtype=torch.long,
        device=device,
    )

    for f_start in tqdm(
        range(0, len(f_idx), FEED_CHUNK),
        desc=f"E5 retrieval — {country}",
        unit="feed chunk",
        dynamic_ncols=True,
    ):
        f_end = min(f_start + FEED_CHUNK, len(f_idx))
        feed_global_idx = f_idx[f_start:f_end]

        # Normalized embeddings were saved as float16, so cosine similarity
        # is just a matrix multiplication.
        feed_gpu = torch.from_numpy(
            np.asarray(feed_embeddings[feed_global_idx], dtype=np.float16)
        ).to(device)

        feed_global_idx_gpu = torch.from_numpy(
            feed_global_idx.astype(np.int64, copy=False)
        ).to(device)

        for q_start in range(0, len(q_idx), QUERY_BATCH):
            q_end = min(q_start + QUERY_BATCH, len(q_idx))
            query_global_idx = q_idx[q_start:q_end]

            query_gpu = torch.from_numpy(
                np.asarray(
                    source1_embeddings[query_global_idx],
                    dtype=np.float16,
                )
            ).to(device)

            # fp16 matmul on the V100; cast scores to fp32 before top-k merging.
            scores = (
                query_gpu @ feed_gpu.T
            ).float()

            chunk_k = min(E5_TOP_K, scores.shape[1])

            chunk_scores, chunk_local_idx = torch.topk(
                scores,
                k=chunk_k,
                dim=1,
            )

            chunk_global_idx = feed_global_idx_gpu[
                chunk_local_idx
            ]

            old_scores = best_scores[q_start:q_end]
            old_indices = best_indices[q_start:q_end]

            merged_scores = torch.cat(
                [old_scores, chunk_scores],
                dim=1,
            )

            merged_indices = torch.cat(
                [old_indices, chunk_global_idx],
                dim=1,
            )

            new_scores, new_pos = torch.topk(
                merged_scores,
                k=E5_TOP_K,
                dim=1,
            )

            new_indices = torch.gather(
                merged_indices,
                1,
                new_pos,
            )

            best_scores[q_start:q_end] = new_scores
            best_indices[q_start:q_end] = new_indices

            del query_gpu, scores
            del chunk_scores, chunk_local_idx, chunk_global_idx
            del merged_scores, merged_indices, new_scores, new_indices

        del feed_gpu, feed_global_idx_gpu
        torch.cuda.empty_cache()

    country_scores = best_scores.cpu().numpy()
    country_feed_idx = best_indices.cpu().numpy()

    for local_q, global_q in enumerate(q_idx):
        for rank in range(E5_TOP_K):
            global_f = int(country_feed_idx[local_q, rank])

            all_result_rows.append({
                "source1_entity_id": source1.iloc[global_q]["entity_id"],
                "candidate_entity_id": feed.iloc[global_f]["entity_id"],
                "e5_similarity": float(country_scores[local_q, rank]),
                "e5_rank": rank + 1,
                "country": country,
            })

    del best_scores, best_indices
    torch.cuda.empty_cache()

e5_candidates = pd.DataFrame(all_result_rows)

E5_CANDIDATE_PATH = (
    REPO
    / "cache"
    / "candidates"
    / "e26_multilingual_e5_top20.parquet"
)

E5_CANDIDATE_PATH.parent.mkdir(parents=True, exist_ok=True)

e5_candidates.to_parquet(
    E5_CANDIDATE_PATH,
    index=False,
)

elapsed = time.time() - t0

print("\n✅ E5 RETRIEVAL COMPLETE")
print("Candidate rows:", f"{len(e5_candidates):,}")
print(
    "Candidates / Source1:",
    f"{len(e5_candidates) / len(source1):.1f}",
)
print("Elapsed:", f"{elapsed / 60:.2f} min")
print("Saved:", E5_CANDIDATE_PATH)

print("\nSimilarity statistics:")
display(
    e5_candidates["e5_similarity"]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .to_frame()
)


🌍 INDIA: 3,962 Source1 × 414,014 feed records


E5 retrieval — india:   0%|          | 0/5 [00:00<?, ?feed chunk/s]


🌍 US: 6,038 Source1 × 620,488 feed records


E5 retrieval — us:   0%|          | 0/7 [00:00<?, ?feed chunk/s]


✅ E5 RETRIEVAL COMPLETE
Candidate rows: 200,000
Candidates / Source1: 20.0
Elapsed: 0.29 min
Saved: /home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/candidates/e26_multilingual_e5_top20.parquet

Similarity statistics:


,e5_similarity
count,200000.000000
mean,0.905980
std,0.022949
min,0.863770
50%,0.900879
90%,0.945312
95%,0.958008
99%,0.966309
max,0.978516


In [47]:
# E2.6 Cell 8: Measure how many true links multilingual-E5 top-20 recovers,
# especially among the true links missed by the existing E2.3 candidate set.

import pandas as pd
import numpy as np

GT_PATH = (
    REPO
    / "student_resource"
    / "dataset"
    / "train"
    / "train_ground_truth.tsv"
)

OLD_CANDIDATE_PATH = (
    REPO
    / "cache"
    / "candidates"
    / "e23_retrieval_38a0b640f78b_selected.parquet"
)

gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str,
)

# Restrict ground truth to the same 10k Source1 validation universe.
valid_s1_ids = set(source1["entity_id"].astype(str))

gt = gt[
    gt["source1_entity_id"].isin(valid_s1_ids)
].copy()

# Expand matched_entity_ids into one row per true pair.
truth_pairs = (
    gt.assign(
        matched_entity_ids=
        gt["matched_entity_ids"]
        .fillna("")
        .str.split()
    )
    .explode("matched_entity_ids")
)

truth_pairs = truth_pairs[
    truth_pairs["matched_entity_ids"].fillna("") != ""
].copy()

truth_pairs = truth_pairs.rename(
    columns={"matched_entity_ids": "candidate_entity_id"}
)

truth_pairs = truth_pairs[
    ["source1_entity_id", "candidate_entity_id"]
].drop_duplicates()

# Load E2.3 candidates.
old_candidates = pd.read_parquet(
    OLD_CANDIDATE_PATH
)

# Robustly identify the target/candidate ID column.
candidate_id_options = [
    "candidate_entity_id",
    "feed_entity_id",
    "target_entity_id",
    "entity_id_candidate",
]

old_candidate_col = next(
    (
        col
        for col in candidate_id_options
        if col in old_candidates.columns
    ),
    None,
)

if old_candidate_col is None:
    raise ValueError(
        "Could not identify candidate ID column. "
        f"Columns are: {old_candidates.columns.tolist()}"
    )

old_pairs = (
    old_candidates[
        ["source1_entity_id", old_candidate_col]
    ]
    .rename(
        columns={
            old_candidate_col: "candidate_entity_id"
        }
    )
    .drop_duplicates()
)

e5_pairs = (
    e5_candidates[
        ["source1_entity_id", "candidate_entity_id"]
    ]
    .drop_duplicates()
)

truth_set = set(
    zip(
        truth_pairs["source1_entity_id"],
        truth_pairs["candidate_entity_id"],
    )
)

old_set = set(
    zip(
        old_pairs["source1_entity_id"],
        old_pairs["candidate_entity_id"],
    )
)

e5_set = set(
    zip(
        e5_pairs["source1_entity_id"],
        e5_pairs["candidate_entity_id"],
    )
)

old_hits = truth_set & old_set
e5_hits = truth_set & e5_set

old_misses = truth_set - old_set

e5_recovered_old_misses = (
    old_misses & e5_set
)

union_hits = truth_set & (
    old_set | e5_set
)

print("TRUE LINKS:", f"{len(truth_set):,}")

print("\nE2.3 retrieval:")
print("Recovered:", f"{len(old_hits):,}")
print(
    "Pair recall:",
    f"{len(old_hits) / len(truth_set):.6%}",
)
print("Missed:", f"{len(old_misses):,}")

print("\nE5 top-20 alone:")
print("Recovered:", f"{len(e5_hits):,}")
print(
    "Pair recall:",
    f"{len(e5_hits) / len(truth_set):.6%}",
)

print("\n🔥 E5 RECOVERY OF E2.3 MISSES:")
print(
    "Recovered old misses:",
    f"{len(e5_recovered_old_misses):,}",
    "/",
    f"{len(old_misses):,}",
)
print(
    "Recovery rate:",
    f"{len(e5_recovered_old_misses) / max(len(old_misses), 1):.2%}",
)

print("\nE2.3 ∪ E5:")
print("Recovered:", f"{len(union_hits):,}")
print(
    "Pair recall:",
    f"{len(union_hits) / len(truth_set):.6%}",
)
print(
    "Remaining misses:",
    f"{len(truth_set - (old_set | e5_set)):,}",
)

# Country-level recovery of previously missed true links.
truth_country = (
    truth_pairs
    .merge(
        source1[
            ["entity_id", "country_norm"]
        ],
        left_on="source1_entity_id",
        right_on="entity_id",
        how="left",
    )
)

truth_country["pair"] = list(
    zip(
        truth_country["source1_entity_id"],
        truth_country["candidate_entity_id"],
    )
)

print("\nRecovery by country:")

for country in sorted(
    truth_country["country_norm"]
    .dropna()
    .unique()
):
    country_truth = set(
        truth_country.loc[
            truth_country["country_norm"] == country,
            "pair",
        ]
    )

    country_old_miss = (
        country_truth - old_set
    )

    country_e5_recovered = (
        country_old_miss & e5_set
    )

    print(
        f"{country:8s} "
        f"old misses={len(country_old_miss):4d} | "
        f"E5 recovered={len(country_e5_recovered):4d} | "
        f"recovery="
        f"{len(country_e5_recovered) / max(len(country_old_miss), 1):.2%}"
    )

TRUE LINKS: 9,405

E2.3 retrieval:
Recovered: 500
Pair recall: 5.316321%
Missed: 8,905

E5 top-20 alone:
Recovered: 468
Pair recall: 4.976077%

🔥 E5 RECOVERY OF E2.3 MISSES:
Recovered old misses: 6 / 8,905
Recovery rate: 0.07%

E2.3 ∪ E5:
Recovered: 506
Pair recall: 5.380117%
Remaining misses: 8,899

Recovery by country:
india    old misses=3534 | E5 recovered=   4 | recovery=0.11%
us       old misses=5371 | E5 recovered=   2 | recovery=0.04%


In [48]:
# E2.6 Cell 8A: Inspect the raw ground-truth encoding and delimiter format,
# because the previous recall cell parsed only 9,405 links instead of ~34,502.

GT_PATH = (
    REPO
    / "student_resource"
    / "dataset"
    / "train"
    / "train_ground_truth.tsv"
)

gt_raw = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str,
)

gt_10k = gt_raw[
    gt_raw["source1_entity_id"].isin(
        set(source1["entity_id"].astype(str))
    )
].copy()

print("10k GT rows:", len(gt_10k))

print("\nRaw matched_entity_ids examples:")
for value in (
    gt_10k["matched_entity_ids"]
    .dropna()
    .head(20)
):
    print(repr(value))

print("\nCharacter diagnostics:")
sample = (
    gt_10k["matched_entity_ids"]
    .dropna()
    .astype(str)
)

for char, label in [
    (",", "comma"),
    ("|", "pipe"),
    (";", "semicolon"),
    (" ", "space"),
    ("[", "left bracket"),
]:
    print(
        f"{label:15s}",
        int(sample.str.contains(char, regex=False).sum()),
    )

print("\nRaw non-empty GT rows:")
print(
    (gt_10k["matched_entity_ids"].fillna("") != "").sum()
)

print("\nPreviously expected from E2.3:")
print("True links: ~34,502")
print("Recovered:  ~33,667")

10k GT rows: 10000

Raw matched_entity_ids examples:
'S2-851752978,S3-286175400,S3-1218889,S3-564988957,S3-841777268'
'S2-238550131,S3-551152157,S3-484968993'
'S2-689298806,S3-49524617,S3-632926404'
'S2-178788281,S2-792185760,S3-652494810,S3-22789765'
'S2-881028278,S3-810547728'
'S2-615990414,S3-305681780'
'S2-397145365,S2-265886457,S3-839447204,S3-780004361'
'S2-454875880,S2-571443346,S2-777135419,S3-387598114,S3-133060203'
'S2-96767082,S3-360367188'
'S2-601177655,S2-809918482,S3-588852284,S3-339503285,S3-744911212'
'S2-870322280,S3-539870164,S3-755224957'
'S2-523234440'
'S3-177986687,S3-233035925,S3-837067038'
'S2-245917164,S3-337696255,S3-547897382,S3-127674718'
'S2-564058539,S3-902112308,S3-315574887,S3-673780219'
'S2-800792433,S2-695180837,S2-658157669,S3-568452335,S3-687481435,S3-706835965'
'S2-52852038,S2-975723017,S3-384926633,S3-735381233'
'S3-740363922'
'S3-377809472,S3-194959047,S3-616448772'
'S2-86133459,S2-521480213,S3-205311812,S3-783457347,S3-22610069'

Character diagnos

In [49]:
# E2.6 Cell 8B: Recompute E2.3 vs multilingual-E5 retrieval recall using the
# correct comma-separated ground-truth format.

import pandas as pd

gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str,
)

valid_s1_ids = set(
    source1["entity_id"].astype(str)
)

gt = gt[
    gt["source1_entity_id"].isin(valid_s1_ids)
].copy()

# Correct parsing: matched IDs are comma-separated.
truth_pairs = (
    gt.assign(
        candidate_entity_id=
        gt["matched_entity_ids"]
        .fillna("")
        .str.split(",")
    )
    .explode("candidate_entity_id")
)

truth_pairs["candidate_entity_id"] = (
    truth_pairs["candidate_entity_id"]
    .fillna("")
    .str.strip()
)

truth_pairs = truth_pairs[
    truth_pairs["candidate_entity_id"] != ""
][
    ["source1_entity_id", "candidate_entity_id"]
].drop_duplicates()

old_candidates = pd.read_parquet(
    OLD_CANDIDATE_PATH
)

candidate_id_options = [
    "candidate_entity_id",
    "feed_entity_id",
    "target_entity_id",
    "entity_id_candidate",
]

old_candidate_col = next(
    col
    for col in candidate_id_options
    if col in old_candidates.columns
)

old_pairs = (
    old_candidates[
        ["source1_entity_id", old_candidate_col]
    ]
    .rename(
        columns={
            old_candidate_col: "candidate_entity_id"
        }
    )
    .drop_duplicates()
)

e5_pairs = (
    e5_candidates[
        ["source1_entity_id", "candidate_entity_id"]
    ]
    .drop_duplicates()
)

truth_set = set(
    zip(
        truth_pairs["source1_entity_id"],
        truth_pairs["candidate_entity_id"],
    )
)

old_set = set(
    zip(
        old_pairs["source1_entity_id"],
        old_pairs["candidate_entity_id"],
    )
)

e5_set = set(
    zip(
        e5_pairs["source1_entity_id"],
        e5_pairs["candidate_entity_id"],
    )
)

old_hits = truth_set & old_set
e5_hits = truth_set & e5_set
old_misses = truth_set - old_set
e5_recovered_old_misses = old_misses & e5_set
union_hits = truth_set & (old_set | e5_set)

print("TRUE LINKS:", f"{len(truth_set):,}")

print("\nE2.3 retrieval:")
print("Recovered:", f"{len(old_hits):,}")
print(
    "Pair recall:",
    f"{len(old_hits) / len(truth_set):.6%}",
)
print("Missed:", f"{len(old_misses):,}")

print("\nE5 top-20 alone:")
print("Recovered:", f"{len(e5_hits):,}")
print(
    "Pair recall:",
    f"{len(e5_hits) / len(truth_set):.6%}",
)

print("\n🔥 E5 RECOVERY OF E2.3 MISSES:")
print(
    "Recovered old misses:",
    f"{len(e5_recovered_old_misses):,}",
    "/",
    f"{len(old_misses):,}",
)
print(
    "Recovery rate:",
    f"{len(e5_recovered_old_misses) / max(len(old_misses), 1):.2%}",
)

print("\nE2.3 ∪ E5:")
print("Recovered:", f"{len(union_hits):,}")
print(
    "Pair recall:",
    f"{len(union_hits) / len(truth_set):.6%}",
)
print(
    "Remaining misses:",
    f"{len(truth_set - (old_set | e5_set)):,}",
)

# Country breakdown.
truth_country = truth_pairs.merge(
    source1[
        ["entity_id", "country_norm"]
    ],
    left_on="source1_entity_id",
    right_on="entity_id",
    how="left",
)

truth_country["pair"] = list(
    zip(
        truth_country["source1_entity_id"],
        truth_country["candidate_entity_id"],
    )
)

print("\nRecovery by country:")

for country in sorted(
    truth_country["country_norm"]
    .dropna()
    .unique()
):
    country_truth = set(
        truth_country.loc[
            truth_country["country_norm"] == country,
            "pair",
        ]
    )

    country_old_miss = country_truth - old_set
    country_e5_recovered = country_old_miss & e5_set

    print(
        f"{country:8s} "
        f"old misses={len(country_old_miss):4d} | "
        f"E5 recovered={len(country_e5_recovered):4d} | "
        f"recovery="
        f"{len(country_e5_recovered) / max(len(country_old_miss), 1):.2%}"
    )

TRUE LINKS: 34,502

E2.3 retrieval:
Recovered: 33,667
Pair recall: 97.579850%
Missed: 835

E5 top-20 alone:
Recovered: 31,934
Pair recall: 92.556953%

🔥 E5 RECOVERY OF E2.3 MISSES:
Recovered old misses: 326 / 835
Recovery rate: 39.04%

E2.3 ∪ E5:
Recovered: 33,993
Pair recall: 98.524723%
Remaining misses: 509

Recovery by country:
india    old misses= 555 | E5 recovered= 139 | recovery=25.05%
us       old misses= 280 | E5 recovered= 187 | recovery=66.79%


In [50]:
# E2.6 Cell 9: Merge E2.3 and multilingual-E5 candidates into one semantic
# candidate union, measure the candidate-count cost, and checkpoint the result.

import numpy as np
import pandas as pd

# Normalize the old candidate ID column.
old_union = old_candidates.copy()

if old_candidate_col != "candidate_entity_id":
    old_union = old_union.rename(
        columns={old_candidate_col: "candidate_entity_id"}
    )

# Mark retrieval provenance.
old_union["retrieved_e23"] = np.uint8(1)

# Keep the E5 score/rank for semantic candidates.
e5_union = e5_candidates[
    [
        "source1_entity_id",
        "candidate_entity_id",
        "e5_similarity",
        "e5_rank",
    ]
].copy()

e5_union["retrieved_e5"] = np.uint8(1)

# Outer merge preserves every candidate found by either retrieval system.
union_candidates = old_union.merge(
    e5_union,
    on=[
        "source1_entity_id",
        "candidate_entity_id",
    ],
    how="outer",
)

union_candidates["retrieved_e23"] = (
    union_candidates["retrieved_e23"]
    .fillna(0)
    .astype(np.uint8)
)

union_candidates["retrieved_e5"] = (
    union_candidates["retrieved_e5"]
    .fillna(0)
    .astype(np.uint8)
)

union_candidates["e5_similarity"] = (
    union_candidates["e5_similarity"]
    .fillna(0.0)
    .astype(np.float32)
)

union_candidates["e5_rank"] = (
    union_candidates["e5_rank"]
    .fillna(0)
    .astype(np.uint16)
)

union_candidates["retrieval_channel_count_e26"] = (
    union_candidates["retrieved_e23"]
    + union_candidates["retrieved_e5"]
).astype(np.uint8)

# Candidate counts per Source1.
candidate_counts = (
    union_candidates
    .groupby("source1_entity_id")
    .size()
)

old_candidate_counts = (
    old_pairs
    .groupby("source1_entity_id")
    .size()
)

print("E2.3 candidate rows:", f"{len(old_pairs):,}")
print("E5 candidate rows:  ", f"{len(e5_pairs):,}")
print("Union rows:          ", f"{len(union_candidates):,}")

print(
    "\nAverage candidates / S1:",
    f"{candidate_counts.mean():.2f}",
)

print(
    "Median candidates / S1:",
    f"{candidate_counts.median():.0f}",
)

print(
    "p95 candidates / S1:",
    f"{candidate_counts.quantile(0.95):.0f}",
)

print(
    "Max candidates / S1:",
    f"{candidate_counts.max():,}",
)

print(
    "\nAverage E2.3 candidates / S1:",
    f"{old_candidate_counts.mean():.2f}",
)

added_rows = len(union_candidates) - len(old_pairs)

print(
    "New unique semantic candidates:",
    f"{added_rows:,}",
)

print(
    "Candidate-set growth:",
    f"{added_rows / len(old_pairs):.2%}",
)

# Provenance breakdown.
provenance = (
    union_candidates
    .groupby(
        ["retrieved_e23", "retrieved_e5"]
    )
    .size()
    .rename("rows")
    .reset_index()
)

print("\nRetrieval provenance:")
display(provenance)

# Checkpoint the union.
E26_UNION_PATH = (
    REPO
    / "cache"
    / "candidates"
    / "e26_e23_plus_e5_top20_union.parquet"
)

union_candidates.to_parquet(
    E26_UNION_PATH,
    index=False,
)

print("\nSaved:")
print(E26_UNION_PATH)

print("\n✅ E2.6 candidate union checkpointed")
print(
    "Known union pair recall:",
    f"{len(union_hits) / len(truth_set):.6%}",
)

E2.3 candidate rows: 488,090
E5 candidate rows:   200,000
Union rows:           609,898

Average candidates / S1: 60.99
Median candidates / S1: 62
p95 candidates / S1: 67
Max candidates / S1: 70

Average E2.3 candidates / S1: 48.81
New unique semantic candidates: 121,808
Candidate-set growth: 24.96%

Retrieval provenance:


,retrieved_e23,retrieved_e5,rows
0,0,1,121808
1,1,0,409898
2,1,1,78192



Saved:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/candidates/e26_e23_plus_e5_top20_union.parquet

✅ E2.6 candidate union checkpointed
Known union pair recall: 98.524723%


In [51]:
# E2.6 Cell 10: Analyze the E5-only candidates to see where the 326 newly
# recovered true links sit by semantic similarity/rank before rebuilding features.

e5_only = union_candidates[
    (union_candidates["retrieved_e23"] == 0)
    & (union_candidates["retrieved_e5"] == 1)
].copy()

truth_pair_index = pd.MultiIndex.from_tuples(
    truth_set,
    names=["source1_entity_id", "candidate_entity_id"],
)

e5_only_index = pd.MultiIndex.from_frame(
    e5_only[
        ["source1_entity_id", "candidate_entity_id"]
    ]
)

e5_only["label"] = (
    e5_only_index.isin(truth_pair_index)
    .astype(np.uint8)
)

print("E5-only candidates:", f"{len(e5_only):,}")
print("E5-only true links:", f"{int(e5_only['label'].sum()):,}")
print(
    "Positive rate:",
    f"{e5_only['label'].mean():.4%}",
)

print("\nSimilarity by label:")
display(
    e5_only.groupby("label")["e5_similarity"]
    .describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

print("\nRecovered true links by E5 rank:")
rank_summary = (
    e5_only[e5_only["label"] == 1]
    .groupby("e5_rank")
    .size()
    .rename("true_links")
    .reset_index()
)

rank_summary["cumulative_true_links"] = (
    rank_summary["true_links"].cumsum()
)

rank_summary["cumulative_recall_of_326"] = (
    rank_summary["cumulative_true_links"]
    / e5_only["label"].sum()
)

display(rank_summary)

print("\nRecovery at rank cutoffs:")

for k in [1, 2, 3, 5, 10, 15, 20]:
    subset = e5_only[
        e5_only["e5_rank"] <= k
    ]

    tp = int(subset["label"].sum())
    rows = len(subset)

    print(
        f"top-{k:2d}: "
        f"recovered={tp:3d}/{int(e5_only['label'].sum())} "
        f"({tp / max(int(e5_only['label'].sum()), 1):6.2%}) | "
        f"candidates={rows:6,d} | "
        f"positive_rate={subset['label'].mean():.4%}"
    )

print("\nTop recovered E5-only positives:")
display(
    e5_only[e5_only["label"] == 1]
    .sort_values(
        ["e5_rank", "e5_similarity"],
        ascending=[True, False],
    )
    [
        [
            "source1_entity_id",
            "candidate_entity_id",
            "e5_similarity",
            "e5_rank",
        ]
    ]
    .head(20)
)

E5-only candidates: 121,808
E5-only true links: 326
Positive rate: 0.2676%

Similarity by label:


,count,mean,std,min,10%,25%,50%,75%,90%,95%,99%,max
label,,,,,,,,,,,,
0,121482.0,0.896157,0.013113,0.863770,0.879883,0.885742,0.895508,0.905273,0.914551,0.919434,0.927246,0.949219
1,326.0,0.917901,0.015784,0.879395,0.898926,0.906860,0.917725,0.927246,0.938232,0.946289,0.956543,0.960938



Recovered true links by E5 rank:


,e5_rank,true_links,cumulative_true_links,cumulative_recall_of_326
0,1,10,10,0.030675
1,2,34,44,0.134969
2,3,47,91,0.279141
3,4,53,144,0.441718
4,5,51,195,0.598160
5,6,34,229,0.702454
6,7,18,247,0.757669
7,8,10,257,0.788344
8,9,4,261,0.800613
9,10,10,271,0.831288



Recovery at rank cutoffs:
top- 1: recovered= 10/326 ( 3.07%) | candidates=   347 | positive_rate=2.8818%
top- 2: recovered= 44/326 (13.50%) | candidates= 1,199 | positive_rate=3.6697%
top- 3: recovered= 91/326 (27.91%) | candidates= 2,978 | positive_rate=3.0557%
top- 5: recovered=195/326 (59.82%) | candidates=10,594 | positive_rate=1.8407%
top-10: recovered=271/326 (83.13%) | candidates=42,872 | positive_rate=0.6321%
top-15: recovered=301/326 (92.33%) | candidates=80,958 | positive_rate=0.3718%
top-20: recovered=326/326 (100.00%) | candidates=121,808 | positive_rate=0.2676%

Top recovered E5-only positives:


,source1_entity_id,candidate_entity_id,e5_similarity,e5_rank
132661,S1-300739721,S3-845099706,0.960938,1
149735,S1-324994194,S2-163255575,0.955078,1
462041,S1-779315483,S2-932253681,0.944824,1
206376,S1-40388359,S3-370007551,0.942383,1
415980,S1-711590211,S3-71571931,0.926758,1
52598,S1-180196064,S2-171885706,0.924805,1
185893,S1-374576758,S3-299638376,0.923340,1
224548,S1-430855633,S2-979233372,0.920410,1
194950,S1-387819994,S3-106357119,0.919922,1
515729,S1-864124252,S3-379579812,0.917969,1


In [52]:
# E2.6 Cell 11: Inspect the currently available notebook functions/variables
# so we can reuse the exact E2.3 feature builder for only the new E5 candidates.

import inspect

print("Candidate feature-related functions currently in memory:\n")

for name, obj in sorted(globals().items()):
    if callable(obj):
        lowered = name.lower()

        if any(
            token in lowered
            for token in [
                "feature",
                "similar",
                "jaccard",
                "candidate",
                "pair",
            ]
        ):
            try:
                signature = str(inspect.signature(obj))
            except Exception:
                signature = "(signature unavailable)"

            print(f"{name}{signature}")

print("\nRelevant DataFrames/arrays currently in memory:\n")

for name, obj in sorted(globals().items()):
    if isinstance(obj, pd.DataFrame):
        lowered = name.lower()

        if any(
            token in lowered
            for token in [
                "feature",
                "candidate",
                "train",
                "pair",
                "source1",
                "feed",
            ]
        ):
            print(
                f"{name:35s} "
                f"shape={obj.shape}"
            )

Candidate feature-related functions currently in memory:


Relevant DataFrames/arrays currently in memory:

X_fold_train                        shape=(295952, 75)
X_train                             shape=(369671, 75)
e5_candidates                       shape=(200000, 5)
e5_pairs                            shape=(200000, 2)
features                            shape=(488090, 78)
feed                                shape=(1034502, 23)
old_candidates                      shape=(488090, 45)
old_pairs                           shape=(488090, 2)
source1                             shape=(10000, 18)
source1_eval                        shape=(10000, 2)
train_entities_df                   shape=(7568, 6)
train_feature_rows                  shape=(369671, 81)
train_gt_counts                     shape=(7568, 3)
truth_pairs                         shape=(34502, 2)
union_candidates                    shape=(609898, 50)


In [53]:
# E2.6 Cell 12: Recover the exact E2.3 feature-building code from this notebook
# so we can reuse the same schema for the 121,808 new E5-only candidates.

import json
from pathlib import Path

NOTEBOOK_PATH = (
    REPO
    / "code"
    / "business_entity_resolution"
    / "notebooks"
    / "05_model.ipynb"
)

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

search_terms = [
    "address_token_jaccard",
    "relative_name_similarity_rank",
    "relative_address_tfidf_rank",
    "features =",
]

matches = []

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(term in source for term in search_terms):
        matches.append((i, source))

print("Matching notebook code cells:", len(matches))

for cell_idx, source in matches:
    print("\n" + "=" * 90)
    print(f"NOTEBOOK CELL INDEX: {cell_idx}")
    print("=" * 90)
    print(source[:12000])

    if len(source) > 12000:
        print("\n... [cell truncated after 12,000 characters] ...")

Matching notebook code cells: 10

NOTEBOOK CELL INDEX: 5
OLD_FEATURE_PATH = (
    REPO
    / "cache"
    / "features"
    / "e21_lgbm_c449c0c6b281_features.parquet"
)

old_features = pd.read_parquet(OLD_FEATURE_PATH)

print("Old E2.1 feature table:", old_features.shape)
print("\nColumns:")
for i, col in enumerate(old_features.columns):
    print(f"{i:02d}: {col}")

display(old_features.head(3))

NOTEBOOK CELL INDEX: 9
# Cell 10: Build the base pairwise feature table for all E2.3 candidate pairs
# using the existing feature-engineering pipeline. This is CPU feature generation,
# not model training yet.

from business_entity_resolution.features import build_feature_table

t0 = time.time()

features = build_feature_table(
    candidates,
    source1,
    feed,
    chunk_size=50_000,
)

elapsed = time.time() - t0

print(f"Base features generated in {elapsed:.1f}s")
print("Feature table shape:", features.shape)
print("Feature columns:", len(features.columns))

display(features.head(3))

NOT

In [54]:
# E2.6 Cell 13: Prepare the E2.3 + E5 candidate union for the original feature
# builder by filling missing legacy/address retrieval metadata with neutral values.

e26_candidates_clean = union_candidates.copy()

# These fields come from the older retrieval system. E5-only candidates never
# passed through those channels, so missing values mean "channel not present".
NEUTRAL_ZERO_COLUMNS = [
    "cheap_score",
    "cheap_score_margin_to_next",
    "rare_name_token_hits",
    "rare_address_token_hits",

    "block_exact_full_name",
    "block_exact_core_name",
    "block_exact_sorted_name",
    "block_rare_name_token",
    "block_exact_address",
    "block_rare_address_token",
    "block_cross_country_exact_name",
    "block_name_tfidf",

    "legacy_cheap_score",
    "is_tfidf_only",
    "is_legacy_only",
    "supported_by_both",
    "legacy_blocking_channel_count",
    "candidate_count_raw",

    # E2.3 address retrieval metadata.
    "address_tfidf_similarity",
    "address_tfidf_rank",
    "address_route_postcode",
    "address_route_postcode_prefix",
    "address_route_numeric",
    "address_route_rare_token",
    "address_route_country_fallback",
    "address_route_count",
    "address_preblock_pool_size",
    "has_address_tfidf",
    "block_address_tfidf",
    "is_address_tfidf_only",
    "retrieval_channel_count",
]

existing_neutral_cols = [
    col
    for col in NEUTRAL_ZERO_COLUMNS
    if col in e26_candidates_clean.columns
]

for col in existing_neutral_cols:
    e26_candidates_clean[col] = (
        e26_candidates_clean[col].fillna(0)
    )

# Preserve semantic provenance explicitly.
e26_candidates_clean["has_e5"] = (
    e26_candidates_clean["retrieved_e5"]
    .fillna(0)
    .astype("uint8")
)

e26_candidates_clean["is_e5_only"] = (
    (
        (e26_candidates_clean["retrieved_e5"] == 1)
        & (e26_candidates_clean["retrieved_e23"] == 0)
    )
    .astype("uint8")
)

e26_candidates_clean["e5_similarity"] = (
    e26_candidates_clean["e5_similarity"]
    .fillna(0)
    .astype("float32")
)

e26_candidates_clean["e5_rank"] = (
    e26_candidates_clean["e5_rank"]
    .fillna(0)
    .astype("uint16")
)

print("Clean union shape:", e26_candidates_clean.shape)
print(
    "E5-only rows:",
    f"{int(e26_candidates_clean['is_e5_only'].sum()):,}",
)

print(
    "Remaining NaNs in neutral retrieval columns:",
    int(
        e26_candidates_clean[
            existing_neutral_cols
        ].isna().sum().sum()
    ),
)

print("\nE5 provenance:")
display(
    e26_candidates_clean[
        [
            "retrieved_e23",
            "retrieved_e5",
            "has_e5",
            "is_e5_only",
            "e5_similarity",
            "e5_rank",
        ]
    ]
    .groupby(
        [
            "retrieved_e23",
            "retrieved_e5",
            "has_e5",
            "is_e5_only",
        ],
        as_index=False,
    )
    .agg(
        rows=("e5_similarity", "size"),
        mean_e5_similarity=("e5_similarity", "mean"),
    )
)

print("\n✅ E2.6 candidate union prepared for feature generation")

Clean union shape: (609898, 52)
E5-only rows: 121,808
Remaining NaNs in neutral retrieval columns: 0

E5 provenance:


,retrieved_e23,retrieved_e5,has_e5,is_e5_only,rows,mean_e5_similarity
0,0,1,1,1,121808,0.896215
1,1,0,0,0,409898,0.000000
2,1,1,1,0,78192,0.921192



✅ E2.6 candidate union prepared for feature generation


In [55]:
# E2.6 Cell 14: Rebuild the original pairwise feature table for the full
# E2.3 + E5 candidate union using the exact same feature-engineering pipeline.

import time
from business_entity_resolution.features import build_feature_table

t0 = time.time()

e26_features = build_feature_table(
    e26_candidates_clean,
    source1,
    feed,
    chunk_size=50_000,
)

elapsed = time.time() - t0

print(f"Base E2.6 features generated in {elapsed:.1f}s")
print("Feature table shape:", e26_features.shape)
print("Feature columns:", len(e26_features.columns))

print("\nSource1 coverage:")
print(
    e26_features["source1_entity_id"].nunique(),
    "/",
    source1["entity_id"].nunique(),
)

print("\nPair count check:")
print("Expected:", f"{len(e26_candidates_clean):,}")
print("Actual:  ", f"{len(e26_features):,}")

assert len(e26_features) == len(e26_candidates_clean)
assert e26_features["source1_entity_id"].nunique() == 10_000

print("\n✅ Base E2.6 feature table built successfully")
display(e26_features.head(3))

ModuleNotFoundError: No module named 'business_entity_resolution'

In [56]:
# E2.6 Cell 14A: Restore the repo's Python package path after the kernel restart
# so the original business_entity_resolution feature-engineering code can import.

import sys
from pathlib import Path

PACKAGE_ROOT = REPO / "code"

if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

from business_entity_resolution.features import build_feature_table

print("✅ Import restored")
print("Package root:", PACKAGE_ROOT)
print("Feature builder:", build_feature_table)

ModuleNotFoundError: No module named 'business_entity_resolution.features'

In [59]:
# E2.6 Cell 14D: Diagnose and clear the stale business_entity_resolution import
# so Python reloads the package from the repo's correct src/ directory.

import sys
import importlib

PACKAGE_SRC = (
    REPO
    / "code"
    / "business_entity_resolution"
    / "src"
)

print("Expected package src:")
print(PACKAGE_SRC)

print("\nBefore cleanup:")
if "business_entity_resolution" in sys.modules:
    pkg = sys.modules["business_entity_resolution"]
    print("Loaded package:", pkg)
    print("Package file:", getattr(pkg, "__file__", None))
    print("Package path:", getattr(pkg, "__path__", None))
else:
    print("business_entity_resolution not currently loaded")

# Remove the package and any submodules from the import cache.
for name in list(sys.modules):
    if (
        name == "business_entity_resolution"
        or name.startswith("business_entity_resolution.")
    ):
        del sys.modules[name]

# Put the correct src-layout root first.
if str(PACKAGE_SRC) in sys.path:
    sys.path.remove(str(PACKAGE_SRC))

sys.path.insert(0, str(PACKAGE_SRC))

importlib.invalidate_caches()

import business_entity_resolution

print("\nAfter cleanup:")
print("Package file:", getattr(business_entity_resolution, "__file__", None))
print("Package path:", list(getattr(business_entity_resolution, "__path__", [])))

from business_entity_resolution.features import build_feature_table

print("\n✅ Correct package imported")
print("Feature builder:", build_feature_table)

Expected package src:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/code/business_entity_resolution/src

Before cleanup:
Loaded package: <module 'business_entity_resolution' (namespace)>
Package file: None
Package path: _NamespacePath(['/home/jupyter-1nt23cb058/Amazon-ML-Challenge/code/business_entity_resolution'])

After cleanup:
Package file: /home/jupyter-1nt23cb058/Amazon-ML-Challenge/code/business_entity_resolution/src/business_entity_resolution/__init__.py
Package path: ['/home/jupyter-1nt23cb058/Amazon-ML-Challenge/code/business_entity_resolution/src/business_entity_resolution']

✅ Correct package imported
Feature builder: <function build_feature_table at 0x7fe7856c2160>


In [60]:
# E2.6 Cell 14: Rebuild the original pairwise feature table for the full
# E2.3 + E5 candidate union using the exact same feature-engineering pipeline.

import time
from business_entity_resolution.features import build_feature_table

t0 = time.time()

e26_features = build_feature_table(
    e26_candidates_clean,
    source1,
    feed,
    chunk_size=50_000,
)

elapsed = time.time() - t0

print(f"Base E2.6 features generated in {elapsed:.1f}s")
print("Feature table shape:", e26_features.shape)
print("Feature columns:", len(e26_features.columns))

print("\nSource1 coverage:")
print(
    e26_features["source1_entity_id"].nunique(),
    "/",
    source1["entity_id"].nunique(),
)

print("\nPair count check:")
print("Expected:", f"{len(e26_candidates_clean):,}")
print("Actual:  ", f"{len(e26_features):,}")

assert len(e26_features) == len(e26_candidates_clean)
assert e26_features["source1_entity_id"].nunique() == 10_000

print("\n✅ Base E2.6 feature table built successfully")
display(e26_features.head(3))

Base E2.6 features generated in 286.9s
Feature table shape: (609898, 53)
Feature columns: 53

Source1 coverage:
10000 / 10000

Pair count check:
Expected: 609,898
Actual:   609,898

✅ Base E2.6 feature table built successfully


,source1_entity_id,candidate_entity_id,name_full_exact,name_core_exact,name_sorted_exact,name_token_jaccard,name_shared_token_count,name_token_count_difference,name_length_ratio,name_character_similarity,...,legacy_cheap_score,is_tfidf_only,is_legacy_only,supported_by_both,fusion_score,fusion_rank,candidate_count_for_s1,cheap_score,cheap_rank,cheap_score_margin_to_next
0,S1-100077788,S2-115845900,0,0,0,0.000000,0,0,0.875000,0.400000,...,0.0,0,0,0,NaN,NaN,NaN,0.0,NaN,0.0
1,S1-100077788,S2-119105481,0,0,0,0.200000,1,2,0.458333,0.457143,...,0.0,1,0,0,0.031250,41.0,50.0,0.0,41.0,0.0
2,S1-100077788,S2-131410081,0,0,0,0.142857,1,0,0.827586,0.264151,...,0.0,0,0,0,0.028571,48.0,50.0,0.0,48.0,0.0
